# Harmonization Metrics Analysis

Comprehensive analysis of batch effect metrics and gene sets across all harmonization attempts.

---

# Part 1 — Metrics Analysis
## §1 — Setup, imports, and data loading

In [ ]:
import json
import math
import re
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import Normalize, to_hex, to_rgba
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu, pearsonr
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
sns.set_style("ticks")

# ── Font size constants (change FONT_BASE to scale all text globally) ──────
FONT_BASE   = 14
FONT_TITLE  = FONT_BASE
FONT_LABEL  = FONT_BASE
FONT_TICK   = FONT_BASE - 2
FONT_LEGEND = FONT_BASE - 2
FONT_ANNOT  = FONT_BASE - 2
FONT_SMALL  = FONT_BASE - 4
FONT_TINY   = FONT_BASE - 4

plt.rcParams.update({
    "svg.fonttype":     "none",
    "figure.dpi":       150,
    "font.size":        FONT_BASE,
    "axes.titlesize":   FONT_TITLE,
    "axes.labelsize":   FONT_LABEL,
    "xtick.labelsize":  FONT_TICK,
    "ytick.labelsize":  FONT_TICK,
    "legend.fontsize":  FONT_LEGEND,
    "figure.titlesize": FONT_TITLE,
})

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)
S3_BUCKET = "$FL_S3_BUCKET"
S3_PREFIX = "FL_batch_correction"


### Cell 1.2 — Load metrics CSV

In [ ]:
df = pd.read_csv("metrics_comprehensive_260513.csv")
df["run_id"] = (
    df["strat"]
    + "__"
    + df["imp"]
    + "__"
    + df["method"]
    + "__post"
    + df["post_rm"].map({False: "0", True: "1"})
)
df = df.set_index("run_id")

META_COLS = ["strat", "imp", "method", "post_rm", "status", "compute_time_s"]
ERROR_COLS = [c for c in df.columns if c.startswith("error")]

df_ok = df[df["status"] == "ok"].copy()
print(
    f"Total rows: {len(df)}  |  OK: {len(df_ok)}  |  Failed: {(df['status']=='failed').sum()}"
)
print(f"Strats: {sorted(df_ok.strat.unique())}")
print(f"Imps: {sorted(df_ok.imp.unique())}")
print(f"Methods ({len(df_ok.method.unique())}): {sorted(df_ok.method.unique())}")
print(f"post_rm values: {sorted(df_ok.post_rm.unique())}")

### Cell 1.3 — Column taxonomy

In [ ]:
GROUP_MAP = {
    "A": ["r2_", "pcr_", "dsc_"],
    "B": ["kbet_", "ilisi_", "clisi_", "asw_batch", "asw_bio", "cms_"],
    "C": ["umap_", "tsne_"],
    "D": ["ks_", "per_gene_batch"],
    "E": [
        "n_samples",
        "n_genes",
        "n_batches",
        "n_cohorts",
        "n_diagnosis_groups",
        "n_samples_per_batch",
        "zero_",
        "fraction_",
        "exp_",
        "per_batch_median",
        "bimodal",
        "below_1",
    ],
    "F": ["vp_"],
    "G": ["graph_"],
    "H": ["avg_intra_", "avg_inter_", "dist_ratio_"],
}

ANNOT_COLS_ORDER = [
    "RNA_BATCH",
    "PLATFORM_RNA",
    "RNASEQ_SOURCE",
    "COHORT_LABEL",
    "Major_group",
    "Diagnosis_cell_type_unified",
    "TUMOR_NORMAL",
    "global/all",
]


def classify_metric_col(col):
    """
    Classify a metric column name into a group letter and annotation column.

    Scans GROUP_MAP in order; the first group whose prefix list contains a
    string that ``col`` starts with (or equals after stripping trailing "_") is
    selected.  If no group matches, returns "?".

    Annotation column is determined by scanning ANNOT_COLS_ORDER and returning
    the first entry that appears as a substring of ``col``; otherwise returns
    "global/all".

    Parameters
    ----------
    col : str
        Metric column name from the metrics CSV.

    Returns
    -------
    tuple of (group, annot_col)
        group     : str — single letter A–H (or "?" if unrecognised)
        annot_col : str — annotation grouping variable (e.g. "RNA_BATCH") or
                    "global/all" if the metric is not tied to a specific covariate
    """
    for g, prefixes in GROUP_MAP.items():
        if any(col.startswith(p) or col == p.rstrip("_") for p in prefixes):
            grp = g
            break
    else:
        grp = "?"
    annot = "global/all"
    for a in ANNOT_COLS_ORDER[:-1]:
        if a in col:
            annot = a
            break
    return grp, annot


_SKIP_EXACT = set(
    [
        "asw_batch_COHORT_LABEL",
        "asw_batch_PLATFORM_RNA",
        "asw_batch_RNASEQ_SOURCE",
        "asw_batch_RNA_BATCH",
        "asw_bio_Diagnosis_cell_type_unified",
        "asw_bio_Major_group",
        "asw_bio_PLATFORM_RNA",
        "asw_bio_TUMOR_NORMAL",
    ]
)

metric_cols = [
    c
    for c in df_ok.columns
    if c not in META_COLS
    and c not in ERROR_COLS
    and c not in _SKIP_EXACT
    and not c.startswith("r2_pc")
    and not c.startswith("dsc_pvalue")
    and not c.startswith("n_cohorts_")
    and not c.startswith("n_samples_per_batch")
    # and not c.startswith("cms_mean")
    and not c.startswith("ilisi_mean")
]

col_meta = pd.DataFrame(
    [classify_metric_col(c) for c in metric_cols],
    index=metric_cols,
    columns=["group", "annot_col"],
)
print(f"Total metric columns: {len(metric_cols)}")
print(col_meta.group.value_counts().sort_index())
# ── Coarse metric type classification ─────────────────────────────────────
METRIC_TYPE_MAP = {
    "local_neighborhood": [
        "kbet_", "ilisi_", "clisi_", "graph_connectivity_",
        "umap_entropy_", "tsne_entropy_",
    ],
    "global_distance": [
        "r2_", "pcr_", "dsc_", "umap_centroid_disp_", "tsne_centroid_disp_",
        "dist_ratio_", "avg_intra_dist_", "avg_inter_dist_",
        "asw_batch_", "asw_bio_", "cms_",
    ],
    "distribution_similarity": [
        "ks_mean_D_", "ks_frac_sig_", "ks_cohort_within_batch_",
        "per_gene_batch_mean_cv_",
    ],
    "other": [],
}


def classify_metric_type(col: str) -> str:
    for type_name, prefixes in METRIC_TYPE_MAP.items():
        if type_name == "other":
            continue
        if any(col.startswith(p) for p in prefixes):
            return type_name
    return "other"


col_meta["metric_type"] = col_meta.index.map(classify_metric_type)
print(col_meta["metric_type"].value_counts())


### Cell 1.4 — Metric polarity (higher=better +1, lower=better -1, descriptive 0)

In [ ]:
# ── Polarity rationale ────────────────────────────────────────────────────────
# POLARITY[c] = +1  → higher raw value = better batch correction or biology
# POLARITY[c] = -1  → lower raw value = better (e.g. lower R² = less batch)
# POLARITY[c] =  0  → descriptive metric: excluded from composite scoring
#
# Group A (r2_, pcr_, dsc_):
#   R² measures fraction of variance explained by batch. Lower is better (-1).
#   PCR (variance-weighted R²): same direction (-1).
#   DSC (distance between batch centroids): lower = centroids more mixed (-1).
#
# Group B (neighbor-based):
#   asw_batch_norm: ASW of batch labels after normalization; higher = better
#     batch mixing in neighbor graph (+1 regardless of which batch column).
#   asw_bio_norm: ASW of biology labels; higher = biology clusters are tight (+1).
#   kbet, ilisi_norm: acceptance rate / local mixing score; higher = better (+1).
#   clisi_mean: local inverse Simpson index for cell type; higher = better (+1).
#   cms_fraction_mixed: fraction of neighbors from different batches; higher = better (+1).
#   avg_intra_dist, avg_inter_dist: raw distances — direction depends on context,
#     so kept as 0 (excluded from scoring). Use dist_ratio instead.
#
# Group C (embedding metrics):
#   entropy_norm: batch entropy in UMAP/tSNE neighborhood; higher = better mixing (+1).
#   centroid_disp: distance between batch centroids in embedding; lower = better (-1).
#
# Group D (distribution):
#   ks_mean_D: mean KS D-statistic; lower = distributions more aligned (-1).
#   per_gene_batch_mean_cv: per-gene CV across batches; lower = less batch CV (-1).
#
# Group E (data quality):
#   n_genes, n_samples: more is better (+1).
#   fraction_cohorts_bimodal: bimodality is a natural gene-expression property,
#     NOT an artifact. Methods that preserve it are preferred (+1).
#   fraction_cohorts_zero_inflated_bimodal: pathological zero inflation = bad (-1).
#   Other descriptive stats (n_batches, exp_*, per_batch_median): polarity 0.
#
# Group F (variancePartition):
#   vp_ for batch columns: lower variance fraction = better batch removal (-1).
#   vp_ for biology columns: higher variance fraction = biology preserved (+1).
#
# Group G (graph connectivity):
#   graph_connectivity_*: higher = biology groups form connected clusters (+1).
#
# Group H (pairwise distances):
#   dist_ratio_*: intra-group mean distance / inter-group mean distance.
#     For batch columns: lower dist_ratio = batches more mixed = better (-1).
#     For biology columns: lower dist_ratio = biology clusters tighter = better (-1).
#   avg_intra_dist, avg_inter_dist: raw distances, not normalized → polarity 0.

POLARITY = {}
for c in metric_cols:
    g, _ = classify_metric_col(c)
    if g == "A":
        POLARITY[c] = -1
    elif g == "B":
        if "asw_bio_norm" in c:
            POLARITY[c] = +1
        elif "asw_batch_norm" in c:
            POLARITY[c] = +1
        elif any(x in c for x in ("kbet", "ilisi_norm", "cms_fraction_mixed")):
            POLARITY[c] = +1
        elif "clisi_mean" in c:
            POLARITY[c] = +1
        else:
            POLARITY[c] = 0
    elif g == "C":
        if "entropy_norm" in c:
            POLARITY[c] = +1
        elif "centroid_disp" in c:
            POLARITY[c] = -1
        else:
            POLARITY[c] = 0
    elif g == "D":
        if any(x in c for x in ("ks_mean_D", "ks_frac_sig", "ks_cohort")):
            POLARITY[c] = -1
        elif "per_gene_batch_mean_cv" in c:
            POLARITY[c] = -1
        else:
            POLARITY[c] = 0
    elif g == "E":
        POLARITY[c] = 0
    elif g == "F":
        POLARITY[c] = -1 if "RNA_BATCH" in c else +1
    elif g == "G":
        POLARITY[c] = +1
    elif g == "H":
        POLARITY[c] = -1 if "dist_ratio" in c else 0
    else:
        POLARITY[c] = 0

# Override pcr_: batch covariates lower=better; biology covariates higher=better
for c in metric_cols:
    if c.startswith("pcr_"):
        if any(
            b in c
            for b in ("RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL")
        ):
            POLARITY[c] = -1
        else:
            POLARITY[c] = +1

scoring_cols = [c for c in metric_cols if POLARITY.get(c, 0) != 0]
desc_cols = [c for c in metric_cols if POLARITY.get(c, 0) == 0]
print(f"Scoring columns: {len(scoring_cols)}  |  Descriptive: {len(desc_cols)}")

In [ ]:
pd.Series(POLARITY).value_counts().sort_index().rename("count").to_frame()

In [ ]:
# ── Manual overrides to POLARITY (applied after the auto-assignment above) ─────
# These entries correct cases where the automatic logic assigns the wrong polarity:
#
# asw_batch_norm_* for non-RNA_BATCH columns:
#   The auto-logic sets asw_batch_norm = +1 only when 'asw_batch_norm' prefix is found.
#   Manual check confirmed that ALL asw_batch_norm columns (regardless of batch column)
#   use the same polarity: higher = better mixing (after normalization).
#
# asw_bio_norm_PLATFORM_RNA = 0:
#   PLATFORM_RNA is a batch/technical column, not a biology column. Measuring
#   ASW of a batch grouping as a "biology" metric is meaningless → excluded.
#
# avg_intra_dist_* = 0, avg_inter_dist_* = 0:
#   Raw distances in high-dimensional space are scale-dependent and not directly
#   comparable across methods (different gene sets → different scale). Use
#   dist_ratio (normalized ratio) for scoring instead.
POLARITY_MANUAL = {
    "asw_batch_norm_COHORT_LABEL": 1,
    "asw_batch_norm_PLATFORM_RNA": 1,
    "asw_batch_norm_RNASEQ_SOURCE": 1,
    "asw_batch_norm_RNA_BATCH": 1,
    "asw_bio_norm_Diagnosis_cell_type_unified": 1,
    "asw_bio_norm_Major_group": 1,
    "asw_bio_norm_PLATFORM_RNA": 0,
    "asw_bio_norm_TUMOR_NORMAL": 1,
    "avg_inter_dist_COHORT_LABEL": 0,
    "avg_inter_dist_Diagnosis_cell_type_unified": 0,
    "avg_inter_dist_Major_group": 0,
    "avg_inter_dist_PLATFORM_RNA": 0,
    "avg_inter_dist_RNASEQ_SOURCE": 0,
    "avg_inter_dist_RNA_BATCH": 0,
    "avg_inter_dist_TUMOR_NORMAL": 0,
    "avg_intra_dist_COHORT_LABEL": 0,
    "avg_intra_dist_Diagnosis_cell_type_unified": 0,
    "avg_intra_dist_Major_group": 0,
    "avg_intra_dist_PLATFORM_RNA": 0,
    "avg_intra_dist_RNASEQ_SOURCE": 0,
    "avg_intra_dist_RNA_BATCH": 0,
    "avg_intra_dist_TUMOR_NORMAL": 0,
    "below_1_fraction_by_batch_max": 0,
    "below_1_fraction_by_batch_min": 0,
    "clisi_mean_Diagnosis_cell_type_unified": 1,
    "clisi_mean_Major_group": 1,
    "clisi_mean_PLATFORM_RNA": 0,
    "clisi_mean_TUMOR_NORMAL": 1,
    "cms_fraction_mixed_COHORT_LABEL": 1,
    "cms_fraction_mixed_PLATFORM_RNA": 1,
    "cms_fraction_mixed_RNASEQ_SOURCE": 1,
    "cms_fraction_mixed_RNA_BATCH": 1,
    "cms_mean_COHORT_LABEL": 0,
    "cms_mean_PLATFORM_RNA": 0,
    "cms_mean_RNASEQ_SOURCE": 0,
    "cms_mean_RNA_BATCH": 0,
    "dist_ratio_COHORT_LABEL": 1,
    "dist_ratio_Diagnosis_cell_type_unified": -1,
    "dist_ratio_Major_group": -1,
    "dist_ratio_PLATFORM_RNA": 1,
    "dist_ratio_RNASEQ_SOURCE": 1,
    "dist_ratio_RNA_BATCH": 1,
    "dist_ratio_TUMOR_NORMAL": -1,
    "dsc_COHORT_LABEL": -1,
    "dsc_PLATFORM_RNA": -1,
    "dsc_RNASEQ_SOURCE": -1,
    "dsc_RNA_BATCH": -1,
    "exp_max": 0,
    "exp_median": 0,
    "exp_min": 0,
    "exp_p01": 0,
    "exp_p05": 0,
    "exp_p25": 0,
    "exp_p75": 0,
    "exp_p95": 0,
    "exp_p99": 0,
    "exp_std": 0,
    "fraction_cohorts_bimodal": 1,
    "fraction_cohorts_zero_inflated_bimodal": 1,
    "fraction_genes_below_1": 0,
    "graph_connectivity_Diagnosis_cell_type_unified": 1,
    "graph_connectivity_Major_group": 1,
    "graph_connectivity_PLATFORM_RNA": 0,
    "graph_connectivity_TUMOR_NORMAL": 1,
    "ilisi_norm_COHORT_LABEL": 1,
    "ilisi_norm_PLATFORM_RNA": 1,
    "ilisi_norm_RNASEQ_SOURCE": 1,
    "ilisi_norm_RNA_BATCH": 1,
    "kbet_acceptance_rate_COHORT_LABEL": 1,
    "kbet_acceptance_rate_PLATFORM_RNA": 1,
    "kbet_acceptance_rate_RNASEQ_SOURCE": 1,
    "kbet_acceptance_rate_RNA_BATCH": 1,
    "ks_cohort_within_batch_frac_sig": -1,
    "ks_cohort_within_batch_mean_D": -1,
    "ks_frac_sig_COHORT_LABEL": -1,
    "ks_frac_sig_PLATFORM_RNA": -1,
    "ks_frac_sig_RNA_BATCH": -1,
    "ks_mean_D_COHORT_LABEL": -1,
    "ks_mean_D_PLATFORM_RNA": -1,
    "ks_mean_D_RNA_BATCH": -1,
    "n_batches": 0,
    "n_cohorts": 0,
    "n_diagnosis_groups": 0,
    "n_genes": 0,
    "n_samples": 0,
    "pcr_COHORT_LABEL": -1,
    "pcr_Diagnosis_cell_type_unified": 1,
    "pcr_Major_group": 1,
    "pcr_PLATFORM_RNA": -1,
    "pcr_RNASEQ_SOURCE": -1,
    "pcr_RNA_BATCH": -1,
    "pcr_TUMOR_NORMAL": 1,
    "per_batch_median_cv": 0,
    "per_gene_batch_mean_cv_COHORT_LABEL": -1,
    "per_gene_batch_mean_cv_PLATFORM_RNA": -1,
    "per_gene_batch_mean_cv_RNA_BATCH": -1,
    "r2_COHORT_LABEL": 0,
    "r2_Major_group": 0,
    "r2_PLATFORM_RNA": 0,
    "r2_RNASEQ_SOURCE": 0,
    "r2_RNA_BATCH": 0,
    "r2_TUMOR_NORMAL": 0,
    "tsne_centroid_disp_COHORT_LABEL": -1,
    "tsne_centroid_disp_PLATFORM_RNA": -1,
    "tsne_centroid_disp_RNASEQ_SOURCE": -1,
    "tsne_centroid_disp_RNA_BATCH": -1,
    "tsne_entropy_mean_COHORT_LABEL": 0,
    "tsne_entropy_mean_PLATFORM_RNA": 0,
    "tsne_entropy_mean_RNASEQ_SOURCE": 0,
    "tsne_entropy_mean_RNA_BATCH": 0,
    "tsne_entropy_norm_COHORT_LABEL": 1,
    "tsne_entropy_norm_PLATFORM_RNA": 1,
    "tsne_entropy_norm_RNASEQ_SOURCE": 1,
    "tsne_entropy_norm_RNA_BATCH": 1,
    "umap_centroid_disp_COHORT_LABEL": -1,
    "umap_centroid_disp_PLATFORM_RNA": -1,
    "umap_centroid_disp_RNASEQ_SOURCE": -1,
    "umap_centroid_disp_RNA_BATCH": -1,
    "umap_entropy_mean_COHORT_LABEL": 0,
    "umap_entropy_mean_PLATFORM_RNA": 0,
    "umap_entropy_mean_RNASEQ_SOURCE": 0,
    "umap_entropy_mean_RNA_BATCH": 0,
    "umap_entropy_norm_COHORT_LABEL": 1,
    "umap_entropy_norm_PLATFORM_RNA": 1,
    "umap_entropy_norm_RNASEQ_SOURCE": 1,
    "umap_entropy_norm_RNA_BATCH": 1,
    "zero_fraction_by_batch_max": 0,
    "zero_fraction_by_batch_min": 0,
    "zero_fraction_global": 0,
}
POLARITY.update(POLARITY_MANUAL)

# Update scoring/desc cols after manual overrides
scoring_cols = [c for c in metric_cols if POLARITY.get(c, 0) != 0]
desc_cols = [c for c in metric_cols if POLARITY.get(c, 0) == 0]
print(
    f"After manual overrides — scoring: {len(scoring_cols)}, descriptive: {len(desc_cols)}"
)

### Cell 1.5 — Normalize metrics to [0,1] where 1=best

In [ ]:
def normalize_for_display(df_vals, cols, polarity_map):
    """
    Normalize metric columns to [0, 1] where 1 always means "best".

    Pipeline: NaN fill → MinMax scaling → polarity flip for lower-is-better metrics.

    Parameters
    ----------
    df_vals : pd.DataFrame
        DataFrame containing raw metric values (samples × metrics).
    cols : list of str
        Metric columns to normalize.  Columns not present in ``df_vals`` are
        silently skipped.
    polarity_map : dict[str, int]
        Maps each column to +1 (higher raw = better) or -1 (lower raw = better).
        Columns with polarity -1 are flipped after scaling so that 1 = best.

    Returns
    -------
    pd.DataFrame
        Same shape as ``df_vals[cols]`` with values in [0, 1] and NaN replaced
        by the column median (neutral position, not best or worst).
    """
    valid_cols = [c for c in cols if c in df_vals.columns]
    normed = df_vals[valid_cols].copy().astype(float)
    # NaN imputation before MinMax scaling: use column median (not mean) to be robust
    # to outlier runs that failed partially (e.g. kBET returned NaN for one imputation).
    # Median-fill ensures the scaling range is not distorted by missing values, and
    # NaN-filled entries are visually indistinguishable from the median — a neutral
    # rather than best/worst position.
    filled = normed.fillna(normed.median())
    scaler = MinMaxScaler()
    normed[valid_cols] = scaler.fit_transform(filled)
    for c in valid_cols:
        if polarity_map.get(c, 1) == -1:
            normed[c] = 1.0 - normed[c]
    return normed

def normalize_for_display_with_dropna(df_vals, cols, polarity_map):
    valid_cols = [c for c in cols if c in df_vals.columns]
    normed = df_vals[valid_cols].copy().astype(float)
    # NaN imputation before MinMax scaling: use column median (not mean) to be robust
    # to outlier runs that failed partially (e.g. kBET returned NaN for one imputation).
    # Median-fill ensures the scaling range is not distorted by missing values, and
    # NaN-filled entries are visually indistinguishable from the median — a neutral
    # rather than best/worst position.
    normed = normed.dropna(thresh=36)
    normed = normed.fillna(0)
    scaler = MinMaxScaler()
    normed[valid_cols] = scaler.fit_transform(normed)
    for c in valid_cols:
        if polarity_map.get(c, 1) == -1:
            normed[c] = 1.0 - normed[c]
    return normed

def dropna_for_display_no_normalization(df_vals, cols, polarity_map):
    valid_cols = [c for c in cols if c in df_vals.columns]
    normed = df_vals[valid_cols].copy().astype(float)
    # NaN imputation before MinMax scaling: use column median (not mean) to be robust
    # to outlier runs that failed partially (e.g. kBET returned NaN for one imputation).
    # Median-fill ensures the scaling range is not distorted by missing values, and
    # NaN-filled entries are visually indistinguishable from the median — a neutral
    # rather than best/worst position.
    normed = normed.dropna(thresh=36)
    normed = normed.fillna(0)
    for c in valid_cols:
        if polarity_map.get(c, 1) == -1:
            normed[c] = 1 - normed[c]
    return normed



df_normed = normalize_for_display_with_dropna(df_ok, scoring_cols, POLARITY)
print("Normalized shape:", df_normed.shape)
print(
    "Value range: [{:.3f}, {:.3f}]".format(df_normed.min().min(), df_normed.max().max())
)

In [ ]:
df_normed

In [ ]:
# Validation: confirm df_normed is correctly shaped and all values in [0, 1]
print(f"Shape: {df_normed[scoring_cols].shape}")
print(
    f"Value range: [{df_normed[scoring_cols].min().min():.4f}, {df_normed[scoring_cols].max().max():.4f}]"
)
nan_count = df_normed[scoring_cols].isna().sum().sum()
print(f"NaN count (should be 0 after fill): {nan_count}")
assert df_normed[scoring_cols].min().min() >= -1e-9, "Values below 0 found"
assert df_normed[scoring_cols].max().max() <= 1 + 1e-9, "Values above 1 found"
assert nan_count == 0, "NaNs remain after normalization"
print("All assertions passed.")

### Cell 1.6 — Multi-level comparison helper

Reusable function used throughout §3. Produces a boxplot + strip (top) and
pairwise FDR-corrected Mann-Whitney p-value heatmap (bottom) for any metric × grouping.

In [ ]:
def plot_multilevel_comparison(
    df: pd.DataFrame,
    metric_col: str,
    group_col: str,
    figsize: tuple = None,
    title: str = None,
    palette: dict = None,
) -> plt.Figure:
    """
    Two-panel figure: boxplot + FDR-corrected significance heatmap.

    Parameters
    ----------
    df : pd.DataFrame
    metric_col : str
    group_col : str
    figsize : tuple, optional
    title : str, optional
    palette : dict, optional
        Color dict keyed by group value. Auto-detected from group_col when None.
    """
    _auto = {"method": method_pal, "strat": strat_pal, "imp": imp_pal}
    if palette is None:
        palette = _auto.get(group_col, {})

    groups = sorted(df[group_col].dropna().unique())
    data_by_group = {
        g: df.loc[df[group_col] == g, metric_col].dropna().values for g in groups
    }
    n = len(groups)
    if figsize is None:
        figsize = (max(8, n * 0.7 + 2), 8)
    fig, (ax_box, ax_pval) = plt.subplots(
        2, 1, figsize=figsize, gridspec_kw={"height_ratios": [3, 1]}
    )
    bp = ax_box.boxplot(
        [data_by_group[g] for g in groups],
        labels=groups,
        patch_artist=True,
        medianprops={"color": "black", "linewidth": 1.5},
    )
    fallback = sns.color_palette("husl", n)
    colors = [palette.get(g, to_hex(fallback[i])) for i, g in enumerate(groups)]
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.55)
    rng = np.random.default_rng(42)
    for i, g in enumerate(groups):
        vals = data_by_group[g]
        jitter = rng.normal(0, 0.08, len(vals))
        ax_box.scatter(
            np.full(len(vals), i + 1) + jitter,
            vals, alpha=0.5, s=8, zorder=5, color=colors[i],
        )
    if title:
        ax_box.set_title(title, fontsize=FONT_TITLE)
    ax_box.set_ylabel(metric_col, fontsize=FONT_LABEL)
    ax_box.set_xticklabels(groups, rotation=45, ha="right", fontsize=FONT_SMALL)

    pmat = np.ones((n, n))
    raw_pvals, pairs = [], []
    for i in range(n):
        for j in range(i + 1, n):
            a, b = data_by_group[groups[i]], data_by_group[groups[j]]
            if len(a) >= 2 and len(b) >= 2:
                _, p = mannwhitneyu(a, b, alternative="two-sided")
                raw_pvals.append(p)
                pairs.append((i, j))
    if raw_pvals:
        _, fdr, _, _ = multipletests(raw_pvals, method="fdr_bh")
        for (i, j), p in zip(pairs, fdr):
            pmat[i, j] = p
            pmat[j, i] = p
    log_pmat = -np.log10(np.clip(pmat, 1e-300, 1.0))
    np.fill_diagonal(log_pmat, 0)
    sns.heatmap(
        log_pmat, ax=ax_pval,
        xticklabels=groups, yticklabels=groups,
        cmap="YlOrRd", vmin=0, linewidths=0.3,
        cbar_kws={"label": "−log₁₀(FDR)", "shrink": 0.6},
    )
    ax_pval.set_xticklabels(groups, rotation=45, ha="right", fontsize=FONT_TINY)
    ax_pval.set_yticklabels(groups, rotation=0, fontsize=FONT_TINY)
    fig.tight_layout()
    return fig


def clustermap_std(
    data: pd.DataFrame,
    metric_colors_df: pd.DataFrame,
    attempt_colors_df: pd.DataFrame,
    row_cluster: bool = True,
    col_cluster: bool = True,
    figsize: tuple = (60, 20),
    title: str = "",
    fname: str = None,
    **kwargs,
):
    """
    Standard clustermap: rows=metrics, cols=attempts.
    metric_colors_df → seaborn row_colors (left).
    attempt_colors_df → seaborn col_colors (top).

    Parameters
    ----------
    data : pd.DataFrame  shape (n_metrics, n_attempts), values in [0, 1]
    metric_colors_df : pd.DataFrame  row annotations
    attempt_colors_df : pd.DataFrame  column annotations
    fname : str or None  saves to FIGURES_DIR/{fname}.svg and .png if given
    """
    # Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations
    g = sns.clustermap(
        data,
        method="ward", metric="euclidean",
        row_cluster=row_cluster, col_cluster=col_cluster,
        row_colors=metric_colors_df.reindex(data.index),
        col_colors=attempt_colors_df.reindex(data.columns),
        cmap="RdYlGn", center=0.5, vmin=0, vmax=1,
        linewidths=0, xticklabels=False, yticklabels=True,
        cbar_pos=(0.02, 0.82, 0.012, 0.12),
        dendrogram_ratio=(0.03, 0.12),
        colors_ratio=(0.015, 0.025),
        **kwargs,
    )
    if title:
        g.fig.suptitle(title, fontsize=FONT_TITLE, y=1.01)
    if fname:
        g.fig.savefig(FIGURES_DIR / f"{fname}.svg", bbox_inches="tight")
        g.fig.savefig(FIGURES_DIR / f"{fname}.png", bbox_inches="tight", dpi=200)
    return g


def add_legend_outside(
    fig: plt.Figure,
    ax: plt.Axes,
    palettes: dict,
    x_offset: float = 1.05,
    fontsize: int = None,
) -> None:
    """
    Add one legend block per palette dict to the right of ax.

    Parameters
    ----------
    palettes : dict of str → dict  (legend title → {label: color})
    """
    if fontsize is None:
        fontsize = FONT_LEGEND
    for idx, (leg_title, pal) in enumerate(palettes.items()):
        handles = make_legend_patches(pal)
        legend = fig.legend(
            handles=handles, title=leg_title,
            loc="right",
            bbox_to_anchor=(x_offset + idx * 0.07, 0.5),
            bbox_transform=ax.transAxes,
            fontsize=fontsize, title_fontsize=fontsize + 1,
            frameon=False,
        )
        fig.add_artist(legend)


def strip_box_plot(
    df: pd.DataFrame,
    x: str,
    y: str,
    ax: plt.Axes,
    palette: dict = None,
    order: list = None,
    box_width: float = 0.5,
    strip_size: float = 4,
    strip_alpha: float = 0.5,
    xlabel: str = None,
    ylabel: str = None,
    title: str = "",
    rotate_x: int = 45,
) -> None:
    """
    Combined boxplot + strip plot. Palette is auto-detected when x is method/strat/imp.
    """
    _auto = {"method": method_pal, "strat": strat_pal, "imp": imp_pal}
    if palette is None:
        palette = _auto.get(x, {})
    if order is None:
        order = sorted(df[x].dropna().unique())
    colors = [palette.get(g, "#AAAAAA") for g in order]
    sns.boxplot(
        data=df, x=x, y=y, ax=ax, order=order,
        palette=dict(zip(order, colors)),
        width=box_width, fliersize=0,
        boxprops={"alpha": 0.6},
    )
    sns.stripplot(
        data=df, x=x, y=y, ax=ax, order=order,
        palette=dict(zip(order, colors)),
        size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
    ax.set_xlabel(xlabel or x, fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel or y, fontsize=FONT_LABEL)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=rotate_x,
                       ha="right", fontsize=FONT_SMALL)
    ax.tick_params(axis="y", labelsize=FONT_TICK)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)


---
## §2 — Full heatmap / clustermap of all metrics

### Build palettes

In [ ]:
import matplotlib.colors as mcolors

ALL_STRATS = [
    "S0_no_removal",
    "A_confirmed_bad",
    "B_extended_bad",
    "C_rnaseq_only",
    "D_malignant_only",
    "E1_iterative_r1",
    "E2_iterative_r2",
    "E3_iterative_r3",
    "F_microarray_only",
    "G_affymetrix_only",
    "H_affymetrix_extended",
]

_pastel_base = sns.color_palette("pastel", n_colors=len(ALL_STRATS))
#strat_pal = dict(zip(ALL_STRATS, _pastel_base))

strat_pal = {
    'S0_no_removal': (0.6313725490196078, 0.788235294117647, 0.9568627450980393),        
    'A_confirmed_bad': (1.0, 0.7058, 0.5098),
    'B_extended_bad': (0.5529, 0.8980, 0.6313),
    'C_rnaseq_only': (1.0, 0.6235, 0.6078),
    'D_malignant_only': (0.8156, 0.7333, 1.0),
    'E1_iterative_r1': (0.8705, 0.7333, 0.6078),
    'E2_iterative_r2': (0.9803, 0.6901, 0.8941),
    'E3_iterative_r3': (0.8117, 0.8117, 0.8117),
    'F_microarray_only': (1.0, 0.9960, 0.6392),
    'G_affymetrix_only': (0.7254, 0.9490, 0.9411),
    'H_affymetrix_extended': (0.741, 0.718, 0.420)
}

imp_pal = {
    "strict":      "#2979ae",
    "knn":         "#982d22",
    "softimpute":  "#713689",
}

ALL_METHODS = sorted(df_ok.method.unique())
_tab20  = sns.color_palette("tab20",  n_colors=20)
_tab20b = sns.color_palette("tab20b", n_colors=20)
_raw_method_colors = (_tab20 + _tab20b)[: max(len(ALL_METHODS), 25)]


def _boost_saturation(rgb, delta: float = 0.15):
    h, s, v = mcolors.rgb_to_hsv(rgb[:3])
    return tuple(mcolors.hsv_to_rgb([h, min(1.0, s + delta), v]))


_vivid_method_colors = [_boost_saturation(c) for c in _raw_method_colors]
method_pal = dict(zip(ALL_METHODS, _vivid_method_colors[: len(ALL_METHODS)]))

post_rm_pal = {False: "#CCCCCC", True: "#333333"}

group_pal = dict(zip("ABCDEFGH", sns.color_palette("Set1", n_colors=8)))

BATCH_ANNOT_COLS  = ["RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL"]
BIO_ANNOT_COLS    = ["Major_group", "Diagnosis_cell_type_unified", "TUMOR_NORMAL"]
GLOBAL_ANNOT_COL  = "global/all"
_batch_pal_colors = ["#8B4513", "#CD853F", "#DEB887", "#F5DEB3"]
_bio_pal_colors   = ["#2F4F4F", "#5F9EA0", "#B0C4DE"]
annot_pal = (
    dict(zip(BATCH_ANNOT_COLS, _batch_pal_colors))
    | dict(zip(BIO_ANNOT_COLS, _bio_pal_colors))
    | {GLOBAL_ANNOT_COL: "#808080"}
)

# ── Metric large-type palette: dark & vivid, distinct from imp_pal ─────────
metric_large_type_pal = {
    "local_neighborhood":    "#1a6b3c",
    "global_distance":       "#8b4a0f",
    "distribution_similarity": "#0d5c63",
    "other":                 "#5c5c1a",
}


def _safe_map(series, pal, default="#AAAAAA"):
    return series.map(pal).fillna(default)


def make_legend_patches(pal_dict):
    return [mpatches.Patch(color=to_hex(v), label=str(k)) for k, v in pal_dict.items()]


# attempt_colors: per-attempt (run) metadata — indexed by run_id
attempt_colors = pd.DataFrame(
    {
        "strat":   _safe_map(df_ok.strat,   strat_pal),
        "imp":     _safe_map(df_ok.imp,     imp_pal),
        "method":  _safe_map(df_ok.method,  method_pal),
        "post_rm": _safe_map(df_ok.post_rm, post_rm_pal),
    },
    index=df_ok.index,
)

# metric_colors: per-metric metadata — indexed by scoring_col name
metric_colors = pd.DataFrame(
    {
        "group":       col_meta.loc[scoring_cols, "group"].map(group_pal).fillna("#AAAAAA"),
        "annot_col":   col_meta.loc[scoring_cols, "annot_col"].map(annot_pal).fillna("#AAAAAA"),
        "metric_type": col_meta.loc[scoring_cols, "metric_type"].map(metric_large_type_pal).fillna("#AAAAAA"),
    },
    index=scoring_cols,
)

print("Palettes built.")
print(f"attempt_colors shape: {attempt_colors.shape}, metric_colors shape: {metric_colors.shape}")


### Visualize palettes

In [ ]:
def plot_palette(palette, title=None):
    """
    Display a color palette as a legend-style swatch figure.

    Parameters
    ----------
    palette : dict
        Mapping of label → color (any matplotlib-compatible color format).
    title : str, optional
        Figure title shown above the swatches.
    """
    fig, ax = plt.subplots(figsize=(4, max(2.5, len(palette) * 0.1)))
    ax.axis("off")
    handles = [
        mpatches.Patch(color=color, label=str(name)) for name, color in palette.items()
    ]
    ax.legend(
        handles=handles,
        loc="center",
        ncol=1,
        fontsize=9,
        frameon=False,
        labelspacing=0.1,
        title=title,
    )
    plt.tight_layout()
    plt.show()


plot_palette(strat_pal, title="strat_pal")

In [ ]:
plot_palette(imp_pal, title="imp_pal")

In [ ]:
plot_palette(method_pal, title="method_pal")

In [ ]:
plot_palette(group_pal, title="group_pal")

In [ ]:
plot_palette(annot_pal, title="annot_pal")
plot_palette(post_rm_pal, title="post_rm_pal")

In [ ]:
plot_palette(metric_large_type_pal, title="metric_large_type_pal")


### Cell 2.2 — Clustered clustermap (rows and columns clustered)

In [ ]:
# ── §2.2 Clustered clustermap ─────────────────────────────────────────────────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

data_for_clustermap = (
    df_normed[scoring_cols].dropna().T
)  # shape: (n_metrics, n_attempts)

g = sns.clustermap(
    data_for_clustermap,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors,    # metric annotations on the LEFT
    col_colors=attempt_colors,   # attempt annotations on TOP
    cmap="RdYlGn",
    center=0.5,
    #vmin=0,
    #vmax=1,
    linewidths=0,
    figsize=(210, 25),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=True,
    yticklabels=True,
    colors_ratio=(0.007, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_ymajorticklabels(), fontsize=16)

legend_specs = [
    ("Strat",       {k: v for k, v in strat_pal.items() if k in df_ok.strat.unique()}),
    ("Imp",         imp_pal),
    ("Method",      {k: v for k, v in method_pal.items() if k in df_ok.method.unique()}),
    ("Post-rm",     post_rm_pal),
    ("Metric group", group_pal),
    ("Annot col",   annot_pal),
    ("Metric type", metric_large_type_pal),
]
add_legend_outside(g.fig, g.ax_heatmap, dict(legend_specs), x_offset=1.07)

g.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_clustered_dropna.pdf", bbox_inches="tight")
g.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_clustered_dropna.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
# ── §2.2b Companion: raw (unnormalized) clustermap ───────────────────────────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

data_for_clustermap_raw = (
    df_ok[scoring_cols].dropna().T
)  # shape: (n_metrics, n_attempts_with_no_NA)

g_raw = sns.clustermap(
    data_for_clustermap_raw,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors.reindex(data_for_clustermap_raw.index),
    col_colors=attempt_colors.reindex(data_for_clustermap_raw.columns),
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(80, 20),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=True,
    colors_ratio=(0.01, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g_raw.ax_heatmap.set_xlabel("")

g_raw.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_raw.pdf", bbox_inches="tight")
g_raw.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_raw.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
df_ok[df_ok.columns[6:]].fillna(0)

In [ ]:
# ALL METRICS including the non-scoring ones
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

data_all = (
    df_ok[df_ok.columns[6:]].apply(pd.to_numeric, errors="coerce").fillna(0).T
)

g_all = sns.clustermap(
    data_all,
    method="ward",
    metric="euclidean",
    row_colors=metric_colors.reindex(data_all.index).fillna("#AAAAAA"),
    col_colors=attempt_colors.reindex(data_all.columns).fillna("#AAAAAA"),
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(80, 20),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=True,
    colors_ratio=(0.01, 0.02),
    cbar_pos=(-0.03, 0.8, 0.015, 0.13),
)
g_all.ax_heatmap.set_xlabel("")

g_all.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_all.pdf", bbox_inches="tight")
g_all.fig.savefig(FIGURES_DIR / "clustermap_all_metrics_all.png", bbox_inches="tight", dpi=200)
plt.show()


### Cell 2.3 — Grouped heatmap (columns ordered by metric group then annot_col; no column clustering)

In [ ]:
# ── §2.3 Grouped heatmap: metrics in fixed group order, attempts clustered ────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

metric_order = (
    col_meta.loc[scoring_cols].sort_values(["annot_col", "group"]).index.tolist()
)

g2 = sns.clustermap(
    data_for_clustermap.reindex(metric_order),
    method="ward",
    metric="euclidean",
    row_cluster=False,
    col_cluster=True,
    row_colors=metric_colors.loc[metric_order],
    col_colors=attempt_colors,
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    figsize=(60, 20),
    dendrogram_ratio=(0.03, 0.12),
    xticklabels=False,
    yticklabels=True,
    colors_ratio=(0.01, 0.02),
)
g2.fig.savefig(FIGURES_DIR / "clustermap_grouped_annot_col.svg", bbox_inches="tight")
g2.fig.savefig(FIGURES_DIR / "clustermap_grouped_metric_order.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
# ── §2.3b Ordered clustermaps: attempts sorted by metadata ───────────────────
# Orientation: rows=metrics, cols=attempts → row_colors=metric annotations, col_colors=attempt annotations

for sort_keys, fname_suffix in [
    (["strat", "imp"],   "by_strat"),
    (["method", "imp"],  "by_method"),
    (["imp", "strat"],   "by_imp"),
]:
    col_order_sorted = (
        df_ok.reindex(data_for_clustermap.columns)
        .sort_values(sort_keys)
        .index.tolist()
    )
    g_sorted = sns.clustermap(
        data_for_clustermap[col_order_sorted],
        method="ward",
        metric="euclidean",
        row_cluster=True,
        col_cluster=False,
        row_colors=metric_colors,
        col_colors=attempt_colors.reindex(col_order_sorted),
        cmap="RdYlGn",
        center=0.5,
        vmin=0,
        vmax=1,
        linewidths=0,
        figsize=(60, 20),
        dendrogram_ratio=(0.03, 0.12),
        xticklabels=False,
        yticklabels=True,
        colors_ratio=(0.01, 0.02),
    )
    g_sorted.fig.savefig(
        FIGURES_DIR / f"clustermap_sorted_{fname_suffix}.svg", bbox_inches="tight"
    )
    g_sorted.fig.savefig(
        FIGURES_DIR / f"clustermap_sorted_{fname_suffix}.png",
        bbox_inches="tight", dpi=200,
    )
    plt.show()


### Cell 2.4 — Ordered heatmap (no clustering — fixed row and column order)

In [ ]:
# ── §2.4 Fully ordered heatmap (no clustering) ────────────────────────────────
# Rows (metrics) ordered: metric group → annotation column.
# Columns (attempts) ordered: strat harshness → imp → method → post_rm.
# This is the most readable structural overview — no dendrograms distort the layout.

METRIC_ORDER = (
    col_meta.loc[scoring_cols].sort_values(["group", "annot_col"]).index.tolist()
)

ATTEMPT_ORDER = (
    df_ok.assign(_si=pd.Categorical(df_ok.strat, categories=ALL_STRATS, ordered=True))
    .sort_values(["_si", "imp", "method", "post_rm"])
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(60, 18))
sns.heatmap(
    df_normed[scoring_cols].dropna().T.reindex(METRIC_ORDER)[ATTEMPT_ORDER],
    ax=ax,
    cmap="RdYlGn",
    center=0.5,
    vmin=0,
    vmax=1,
    linewidths=0,
    xticklabels=False,
    yticklabels=True,
    cbar_kws={"shrink": 0.3, "label": "Normalized score (1=best)"},
)
ax.set_title(
    "Ordered overview — cols: strat\u2192imp\u2192method\u2192post_rm; rows: metric group\u2192annot_col"
)
fig.savefig(FIGURES_DIR / "heatmap_ordered_no_clustering.svg", bbox_inches="tight")
fig.savefig(
    FIGURES_DIR / "heatmap_ordered_no_clustering.png", bbox_inches="tight", dpi=200
)
plt.show()

### §2.5 — Method similarity: clustermaps, embeddings, network


In [ ]:
# ── §2.5.1a Spearman correlation between scoring metrics ─────────────────────
# Orientation: rows=metrics, cols=metrics — same metric_colors on BOTH axes
metric_corr = df_ok[scoring_cols].corr(method="spearman")

g_mcorr = sns.clustermap(
    metric_corr,
    method="ward", metric="euclidean",
    row_colors=metric_colors,
    col_colors=metric_colors,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0, figsize=(28, 26),
    xticklabels=True, yticklabels=True,
    annot_kws={"size": FONT_TINY},
)
g_mcorr.fig.suptitle("Metric–metric Spearman ρ (same metric_colors on both axes)",
                     fontsize=FONT_TITLE, y=1.01)
g_mcorr.fig.savefig(FIGURES_DIR / "metric_correlation_clustermap.svg", bbox_inches="tight")
g_mcorr.fig.savefig(FIGURES_DIR / "metric_correlation_clustermap.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
att_score_mat.sum(axis=1)

In [ ]:
# ── §2.5.1b Spearman correlation between harmonization attempts ──────────────
# Orientation: rows=attempts, cols=attempts — attempt_colors on BOTH axes
attempt_ids   = df_ok.index.tolist()
att_score_mat = df_ok[scoring_cols].fillna(0)
att_score_mat = att_score_mat[att_score_mat.sum(axis=1)!=0]
attempt_corr  = att_score_mat.T.corr(method="spearman")

_att_col_idx = attempt_colors.reindex(attempt_corr.index)

g_acorr = sns.clustermap(
    attempt_corr,
    method="ward", metric="euclidean",
    row_colors=_att_col_idx,
    col_colors=_att_col_idx,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0, figsize=(28, 26),
    xticklabels=False, yticklabels=False,
)
g_acorr.fig.suptitle("Attempt–attempt Spearman ρ across all metrics",
                     fontsize=FONT_TITLE, y=1.01)
g_acorr.fig.savefig(FIGURES_DIR / "attempt_correlation_clustermap.pdf", bbox_inches="tight")
g_acorr.fig.savefig(FIGURES_DIR / "attempt_correlation_clustermap.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
# add customizable other palettes and fields here

In [ ]:
# ── §2.5.2 Embeddings of attempts and metrics in each other's space ──────────
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.decomposition import PCA as _PCA2
import umap as _umap_lib
from sklearn.manifold import TSNE as _TSNE

X_att = _SS().fit_transform(df_ok[scoring_cols].fillna(0).values)
X_met = _SS().fit_transform(df_ok[scoring_cols].fillna(0).T.values)

pca_att  = _PCA2(n_components=2, random_state=42).fit_transform(X_att)
umap_att = _umap_lib.UMAP(n_neighbors=10, min_dist=0.2, random_state=42).fit_transform(X_att)
tsne_att = _TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_att)

_perp_met = min(30, len(scoring_cols) - 1)
pca_met  = _PCA2(n_components=2, random_state=42).fit_transform(X_met)
umap_met = _umap_lib.UMAP(n_neighbors=10, min_dist=0.2, random_state=42).fit_transform(X_met)
tsne_met = _TSNE(n_components=2, perplexity=_perp_met, random_state=42).fit_transform(X_met)

ATT_COORDS = [pca_att,  umap_att,  tsne_att]
ATT_LABELS = [("PC1","PC2"), ("UMAP1","UMAP2"), ("tSNE1","tSNE2")]
MET_COORDS = [pca_met,  umap_met,  tsne_met]
MET_LABELS = [("PC1","PC2"), ("UMAP1","UMAP2"), ("tSNE1","tSNE2")]

fig, axes = plt.subplots(2, 3, figsize=(24, 14))

for col, (coords, (xl, yl)) in enumerate(zip(ATT_COORDS, ATT_LABELS)):
    ax = axes[0, col]
    for method_name in df_ok.method.unique():
        rows = df_ok.index.get_indexer(df_ok[df_ok.method == method_name].index)
        rows = rows[rows >= 0]
        ax.scatter(coords[rows, 0], coords[rows, 1],
                   c=method_pal.get(method_name, "#AAAAAA"),
                   label=method_name, s=30, alpha=0.7, linewidths=0, rasterized=True)
    ax.set_xlabel(xl, fontsize=FONT_LABEL)
    ax.set_ylabel(yl, fontsize=FONT_LABEL)
    ax.set_title(f"Attempts / method  ({xl[:3]})", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)

metric_types = [
    next((t for t, pfxs in METRIC_TYPE_MAP.items()
          if t != "other" and any(m.startswith(p) for p in pfxs)), "other")
    for m in scoring_cols
]
for col, (coords, (xl, yl)) in enumerate(zip(MET_COORDS, MET_LABELS)):
    ax = axes[1, col]
    for i, (m, mt) in enumerate(zip(scoring_cols, metric_types)):
        ax.scatter(coords[i, 0], coords[i, 1],
                   c=metric_large_type_pal[mt], s=60, alpha=0.8, linewidths=0, zorder=3)
    #    ax.text(coords[i, 0], coords[i, 1], m, fontsize=FONT_TINY, alpha=0.75)
    ax.set_xlabel(xl, fontsize=FONT_LABEL)
    ax.set_ylabel(yl, fontsize=FONT_LABEL)
    ax.set_title(f"Metrics / type  ({xl[:3]})", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)

fig.suptitle("Method and metric similarity embeddings", fontsize=FONT_TITLE + 2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "method_metric_embeddings_2x3.pdf", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "method_metric_embeddings_2x3.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
# ── §2.5.3 Method correlation network ────────────────────────────────────────
try:
    import networkx as nx
    method_profiles = df_ok.groupby("method")[scoring_cols].mean()
    method_corr_net = method_profiles.T.corr(method="spearman")

    G = nx.Graph()
    threshold = 0.75
    for i, m1 in enumerate(method_corr_net.index):
        G.add_node(m1)
        for j, m2 in enumerate(method_corr_net.columns):
            if i < j and method_corr_net.loc[m1, m2] > threshold:
                G.add_edge(m1, m2, weight=float(method_corr_net.loc[m1, m2]))

    pos = nx.spring_layout(G, seed=42, weight="weight")
    fig, ax = plt.subplots(figsize=(12, 10))
    nx.draw_networkx_nodes(G, pos, ax=ax,
        node_color=[method_pal.get(n, "#AAAAAA") for n in G.nodes],
        node_size=600, alpha=0.9)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.4, width=1.5)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=FONT_SMALL)
    ax.set_title(f"Method similarity network (Spearman ρ > {threshold})",
                 fontsize=FONT_TITLE)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "method_similarity_network.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "method_similarity_network.png", bbox_inches="tight", dpi=200)
    plt.show()
except ImportError:
    print("networkx not installed — run: pip install networkx")


---
## §3 — Group-by-group comparative analysis

**Repeated pattern throughout §3:** For every quantitative comparison, three flavors of
`plot_multilevel_comparison` are produced: grouped by (1) harmonization method, (2) removal
strategy, (3) imputation approach. Each includes an FDR-corrected Mann-Whitney p-value heatmap.

### §3.1 — Group A: PCA Variance Decomposition

In [ ]:
ANNOT_COLS_ORDER

#### Mean R2 by columns

In [ ]:
r2_cols_main = ["r2_RNA_BATCH", "r2_Major_group", "r2_TUMOR_NORMAL", "r2_PLATFORM_RNA"]
# r2_cols_main = ["r2_RNA_BATCH", "r2_Major_group", "r2_COHORT_LABEL","r2_RNASEQ_SOURCE","r2_TUMOR_NORMAL", "r2_PLATFORM_RNA"]
r2_cols_present = [c for c in r2_cols_main if c in df_ok.columns]
method_order = (
    df_ok.groupby("method")["r2_RNA_BATCH"].mean().sort_values().index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(method_order))
width = 0.2
palette = sns.color_palette("husl", len(r2_cols_present))
for i, col in enumerate(r2_cols_present):
    means = [df_ok[df_ok.method == m][col].mean() for m in method_order]
    stds = [df_ok[df_ok.method == m][col].std() for m in method_order]
    ax.bar(
        x + i * width,
        means,
        width,
        label=col,
        color=palette[i],
        alpha=0.8,
        yerr=stds,
        capsize=2,
    )
ax.axhline(
    0.20,
    color="red",
    linestyle="--",
    linewidth=1.0,
    label="Target r\u00b2=0.20 (RNA_BATCH)",
)
ax.set_xticks(x + width * (len(r2_cols_present) - 1) / 2)
ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Mean R\u00b2")
ax.set_title("Mean R\u00b2 per covariate \u00d7 method (sorted by r2_RNA_BATCH)")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "a_r2_by_method_covariate.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "a_r2_by_method_covariate.png", bbox_inches="tight", dpi=200)
plt.show()

#### PCR (principal component regression) by columns

In [ ]:
annot_pal

In [ ]:
r2_cols_main[0][4:]

In [ ]:
r2_cols_main = [
    "pcr_COHORT_LABEL",
    "pcr_Diagnosis_cell_type_unified",
    #"pcr_Major_group",
    #"pcr_PLATFORM_RNA",
    #"pcr_RNASEQ_SOURCE",
    "pcr_RNA_BATCH",
    #"pcr_TUMOR_NORMAL",
]
r2_cols_present = [c for c in r2_cols_main if c in df_ok.columns]
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(method_order))
width = 0.2

palette = {column: annot_pal.get(column[4:]) for column in r2_cols_main}
# palette = sns.color_palette("husl", len(r2_cols_present))
for i, col in enumerate(r2_cols_present):
    means = [df_ok[df_ok.method == m][col].mean() for m in method_order]
    stds = [df_ok[df_ok.method == m][col].std() for m in method_order]
    ax.bar(
        x + i * width,
        means,
        width,
        label=col,
        color=palette[col],
        alpha=0.8,
        yerr=stds,
        capsize=2,
    )
ax.axhline(
    0.20,
    color="red",
    linestyle="--",
    linewidth=1.0,
    label="Target r\u00b2=0.20 (RNA_BATCH)",
)
ax.set_xticks(x + width * (len(r2_cols_present) - 1) / 2)
ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("PCR\u00b2")
ax.set_title("PCR\u00b2 per covariate \u00d7 method (sorted by pcr_RNA_BATCH)")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pcr_by_method_covariate.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "pcr_by_method_covariate.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
r2_cols_main = [
    "pcr_COHORT_LABEL",
    "pcr_Diagnosis_cell_type_unified",
    #"pcr_Major_group",
    #"pcr_PLATFORM_RNA",
    #"pcr_RNASEQ_SOURCE",
    "pcr_RNA_BATCH",
    #"pcr_TUMOR_NORMAL",
]
r2_cols_present = [c for c in r2_cols_main if c in df_ok.columns]
method_order = (
    df_ok.groupby("strat")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(method_order))
width = 0.2

palette = {column: annot_pal.get(column[4:]) for column in r2_cols_main}
# palette = sns.color_palette("husl", len(r2_cols_present))
for i, col in enumerate(r2_cols_present):
    means = [df_ok[df_ok.strat == m][col].mean() for m in method_order]
    stds = [df_ok[df_ok.strat == m][col].std() for m in method_order]
    ax.bar(
        x + i * width,
        means,
        width,
        label=col,
        color=palette[col],
        alpha=0.8,
        yerr=stds,
        capsize=2,
    )
ax.axhline(
    0.20,
    color="red",
    linestyle="--",
    linewidth=1.0,
    label="Target =0.20 ",
)
ax.set_xticks(x + width * (len(r2_cols_present) - 1) / 2)
ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("PCR\u00b2")
ax.set_title("PCR\u00b2 per covariate \u00d7 strategy (sorted by pcr_RNA_BATCH)")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pcr_by_strat_covariate.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "pcr_by_strat_covariate.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
method_order = (
    df_ok.groupby("method")["r2_RNA_BATCH"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=df_ok,
    x="method",
    y="r2_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.4,
    order=method_order,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "r² RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle("r² RNA_BATCH by method × imputation", y=1.02, fontsize=FONT_TITLE)
g.figure.savefig(FIGURES_DIR / "a_r2_catplot_by_imp.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_r2_catplot_by_imp.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=df_ok,
    x="method",
    y="pcr_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.4,
    order=method_order,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "PCR RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle("PCR RNA_BATCH by method × imputation", y=1.02, fontsize=FONT_TITLE)
g.figure.savefig(FIGURES_DIR / "a_pcr_catplot_by_imp.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_pcr_catplot_by_imp.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=df_ok,
    x="method",
    y="pcr_RNA_BATCH",
    col="post_rm",
    kind="box",
    height=4,
    aspect=1.4,
    order=method_order,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "PCR RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle("PCR RNA_BATCH by method × post_rm", y=1.02, fontsize=FONT_TITLE)
g.figure.savefig(FIGURES_DIR / "a_pcr_catplot_by_post_rm.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_pcr_catplot_by_post_rm.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# add here a color palette by harshness

for group_col in ["method", "strat", "imp"]:
    fig = plot_multilevel_comparison(
        df_ok, "r2_RNA_BATCH", group_col, title=f"r\u00b2 RNA_BATCH by {group_col}"
    )
    fname = f"a_r2_multilevel_{group_col}"
    fig.savefig(FIGURES_DIR / f"{fname}.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{fname}.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
for group_col in ["method", "strat", "imp"]:
    fig = plot_multilevel_comparison(
        df_ok, "pcr_RNA_BATCH", group_col, title=f"PCR RNA_BATCH by {group_col}"
    )
    fname = f"a_pcr_multilevel_{group_col}"
    fig.savefig(FIGURES_DIR / f"{fname}.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{fname}.png", bbox_inches="tight", dpi=200)
    plt.show()

#### PCR negatively correlates with R2 by RNA_BATCH

In [ ]:
plot_multilevel_comparison(
        df_ok, "pcr_RNA_BATCH", "method", title=f"PCR RNA_BATCH by {group_col}"
    )



In [ ]:
plot_multilevel_comparison(
        df_ok, "r2_RNA_BATCH", "method", title=f"r\u00b2 RNA_BATCH by {group_col}"
    )

In [ ]:
pcr_cols = [c for c in df_ok.columns if c.startswith("pcr_")]

fig, axes = plt.subplots(1, min(len(pcr_cols), 4), figsize=(16, 4), sharey=False)
for ax, col in zip(axes, pcr_cols[:4]):
    for strat_name in df_ok.strat.unique():
        sub = df_ok[df_ok.strat == strat_name]
        ax.scatter(
            sub.method.map({m: i for i, m in enumerate(method_order)}),
            sub[col],
            label=strat_name,
            alpha=0.7,
            s=20,
            color=strat_pal.get(strat_name, "#AAAAAA"),
        )
    ax.set_xticks(range(len(method_order)))
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=7)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("PCR")
axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "a_pcr_strip.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "a_pcr_strip.png", bbox_inches="tight", dpi=200)
plt.show()

for group_col in ["method", "strat", "imp"]:
    fig = plot_multilevel_comparison(
        df_ok, "pcr_RNA_BATCH", group_col, title=f"PCR RNA_BATCH by {group_col}"
    )
    fig.savefig(FIGURES_DIR / f"a_pcr_multilevel_{group_col}.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / f"a_pcr_multilevel_{group_col}.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
df_ok

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
        data=df_ok, x='pcr_COHORT_LABEL', y='pcr_RNA_BATCH', ax=ax,
        hue='method', palette=method_pal
        #palette=dict(zip(order, colors)),
        #size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
        data=df_ok, x='r2_RNA_BATCH', y='pcr_RNA_BATCH', ax=ax,
        hue='method', palette=method_pal
        #palette=dict(zip(order, colors)),
        #size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
        data=df_ok, x='r2_RNA_BATCH', y='pcr_RNA_BATCH', ax=ax,
        hue='strat', palette=strat_pal
        #palette=dict(zip(order, colors)),
        #size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)


In [ ]:
df_ok

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
        data=df_ok, x='r2_RNA_BATCH', y='pcr_RNA_BATCH', ax=ax,
        hue='post_rm', palette=post_rm_pal
        #palette=dict(zip(order, colors)),
        #size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]

pc_r2_cols = [
    f"r2_pc{i}_RNA_BATCH"
    for i in range(1, 11)
    if f"r2_pc{i}_RNA_BATCH" in df_ok.columns
]
if pc_r2_cols:
    pivot = df_ok.groupby("method")[pc_r2_cols].mean().reindex(method_order)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(10, 10))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Reds",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Mean R\u00b2 (RNA_BATCH)"},
    )
    ax.set_title("R\u00b2 explained by RNA_BATCH per PC (mean across imputations)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
method_order = (
    df_ok.groupby("strat")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]

pc_r2_cols = [
    f"r2_pc{i}_RNA_BATCH"
    for i in range(1, 11)
    if f"r2_pc{i}_RNA_BATCH" in df_ok.columns
]
if pc_r2_cols:
    pivot = df_ok.groupby("strat")[pc_r2_cols].mean().reindex(method_order)
    pivot.columns = [f"PC{i}" for i in range(1, len(pivot.columns) + 1)]
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="Reds",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Mean R\u00b2 (RNA_BATCH)"},
    )
    ax.set_title("R\u00b2 explained by RNA_BATCH per PC (mean across imputations)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap_strat.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "a_per_pc_r2_heatmap_strat.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
method_order = (
    df_ok.groupby("method")["pcr_RNA_BATCH"].mean().sort_values().index.tolist()
)[::-1]


if "dsc_RNA_BATCH" in df_ok.columns and "dsc_COHORT_LABEL" in df_ok.columns:
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(
        df_ok["dsc_RNA_BATCH"],
        df_ok["dsc_COHORT_LABEL"],
        s=df_ok["r2_RNA_BATCH"] * 200 + 20,
        c=df_ok.method.map({m: i for i, m in enumerate(method_order)}),
        cmap="tab20",
        alpha=0.7,
    )
    for method_name in method_order:
        sub = df_ok[df_ok.method == method_name]
        ax.annotate(
            method_name,
            (sub["dsc_RNA_BATCH"].mean(), sub["dsc_COHORT_LABEL"].mean()),
            fontsize=6,
            alpha=0.8,
        )
    ax.set_xlabel("DSC RNA_BATCH (\u2193 better)")
    ax.set_ylabel("DSC COHORT_LABEL (\u2193 better)")
    ax.set_title(
        "DSC: global batch vs cohort-level effects — size \u221d r\u00b2 RNA_BATCH"
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "a_dsc_scatter.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "a_dsc_scatter.png", bbox_inches="tight", dpi=200)
    plt.show()

### §3.2 — Group B: Neighbor-based Integration

In [ ]:
kbet_col = "kbet_acceptance_rate_RNA_BATCH"
ilisi_col = "ilisi_norm_RNA_BATCH"
if kbet_col in df_ok.columns and ilisi_col in df_ok.columns:
    imp_size_map = {"strict": 40, "knn": 80, "softimpute": 140}
    fig, ax = plt.subplots(figsize=(10, 7))
    for method_name in method_order:
        sub = df_ok[df_ok.method == method_name]
        for imp_name in sub.imp.unique():
            row = sub[sub.imp == imp_name]
            ax.scatter(
                row[kbet_col],
                row[ilisi_col],
                #s=imp_size_map.get(imp_name, 60),
                color=method_pal.get(method_name, "#AAAAAA"),
                label=method_name if imp_name == sub.imp.iloc[0] else "",
                alpha=0.75,
                edgecolors="white",
                linewidths=0.3,
            )
    ax.set_xlabel("kBET acceptance rate RNA_BATCH (\u2191 better)")
    ax.set_ylabel("iLISI norm RNA_BATCH (\u2191 better)")
    ax.set_title("kBET vs iLISI — size=imputation, color=method")
    # imp_legend = [mpatches.Patch(color="gray", label=f"{k} (s\u2248{v}"))
    #               for k, v in imp_size_map.items()]
    # ax.legend(handles=imp_legend, title="Imputation", loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "b_kbet_ilisi_scatter.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "b_kbet_ilisi_scatter.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
kbet_col = "kbet_acceptance_rate_RNA_BATCH"
ilisi_col = "ilisi_norm_RNA_BATCH"
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
        data=df_ok, x=kbet_col, y=ilisi_col, ax=ax,
        hue='strat', palette=strat_pal
        #palette=dict(zip(order, colors)),
        #size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

#### Barplots by local metrics

In [ ]:
len('kbet_acceptance_rate_')

In [ ]:
r2_cols_main = [
    "kbet_acceptance_rate_RNA_BATCH",
    "kbet_acceptance_rate_COHORT_LABEL",
    "kbet_acceptance_rate_PLATFORM_RNA",
    #"pcr_PLATFORM_RNA",
    #"pcr_RNASEQ_SOURCE",
    #"clisi_mean_Diagnosis_cell_type_unified",
    #"pcr_TUMOR_NORMAL",
]
r2_cols_present = [c for c in r2_cols_main if c in df_ok.columns]
method_order = (
    df_ok.groupby("method")["kbet_acceptance_rate_RNA_BATCH"].mean().sort_values().index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(method_order))
width = 0.3

palette = {column: annot_pal.get(column[21:]) for column in r2_cols_main}
# palette = sns.color_palette("husl", len(r2_cols_present))
for i, col in enumerate(r2_cols_present):
    means = [df_ok[df_ok.method == m][col].mean() for m in method_order]
    stds = [df_ok[df_ok.method == m][col].std() for m in method_order]
    ax.bar(
        x + i * width,
        means,
        width,
        label=col,
        color=palette[col],
        alpha=0.8,
        yerr=stds,
        capsize=2,
    )
ax.set_xticks(x + width * (len(r2_cols_present) - 1) / 2)
ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("PCR\u00b2")
ax.set_title("PCR\u00b2 per covariate \u00d7 method (sorted by pcr_RNA_BATCH)")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "kbet_by_method_covariate.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "kbet_by_method_covariate.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains('LISI')]

In [ ]:
method_order = (
    df_ok.groupby("method")["kbet_acceptance_rate_RNA_BATCH"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=df_ok,
    x="method",
    y="kbet_acceptance_rate_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=method_order,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "kBET RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle("kBET RNA_BATCH by method × imputation", y=1.02, fontsize=FONT_TITLE)
g.figure.savefig(FIGURES_DIR / "a_kbet_catplot_by_imp.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_kbet_catplot_by_imp.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
strat_order = (
    df_ok.groupby("strat")["kbet_acceptance_rate_RNA_BATCH"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=df_ok,
    x="strat",
    y="kbet_acceptance_rate_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=strat_order,
    palette=strat_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "kBET RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle("kBET RNA_BATCH by strat × imputation", y=1.02, fontsize=FONT_TITLE)
g.figure.savefig(FIGURES_DIR / "a_kbet_catplot_by_imp_strat.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_kbet_catplot_by_imp_strat.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
df_ok.columns[df_ok.columns.str.contains('lisi')]

In [ ]:
data_to_show = df_ok[~df_ok.method.isin(['34_arsyn', '38_harman'])]


method_order = (
    data_to_show.groupby("method")["ilisi_mean_RNA_BATCH"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=data_to_show,
    x="method",
    y="ilisi_mean_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=method_order,
    palette=method_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "ilisi RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle("ilisi RNA_BATCH by method × imputation", y=1.02, fontsize=FONT_TITLE)
g.figure.savefig(FIGURES_DIR / "a_ilisi_catplot_by_imp.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_ilisi_catplot_by_imp.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
data_to_show = df_ok[~df_ok.method.isin(['34_arsyn', '38_harman'])]


strat_order = (
    data_to_show.groupby("strat")["ilisi_mean_RNA_BATCH"].mean().sort_values().index.tolist()
)

g = sns.catplot(
    data=data_to_show,
    x="strat",
    y="ilisi_mean_RNA_BATCH",
    col="imp",
    kind="box",
    height=4,
    aspect=1.5,
    order=strat_order,
    palette=strat_pal,
)
g.set_xticklabels(rotation=45, ha="right", fontsize=FONT_SMALL)
g.set_titles("{col_name}", size=FONT_TITLE)
g.set_axis_labels("Method", "ilisi RNA_BATCH", fontsize=FONT_LABEL)
g.figure.suptitle("ilisi RNA_BATCH by strat × imputation", y=1.02, fontsize=FONT_TITLE)
g.figure.savefig(FIGURES_DIR / "a_ilisi_catplot_by_imp_strat.svg", bbox_inches="tight")
g.figure.savefig(FIGURES_DIR / "a_ilisi_catplot_by_imp_strat.png", bbox_inches="tight", dpi=200)
plt.show()

#### Correlations of local and global metrics

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))

g = sns.scatterplot(
        data=df_ok, x='pcr_RNA_BATCH', y='kbet_acceptance_rate_RNA_BATCH', ax=ax,
        hue='method', palette=method_pal
        #palette=dict(zip(order, colors)),
        #size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)


In [ ]:
#fig, ax = plt.subplots(figsize=(7.5, 7.5))


data_to_plot = df_ok[['clisi_mean_Diagnosis_cell_type_unified', 'clisi_mean_Major_group',
 'clisi_mean_TUMOR_NORMAL',
       'ilisi_mean_COHORT_LABEL', 'ilisi_mean_PLATFORM_RNA',
       'ilisi_mean_RNASEQ_SOURCE', 'ilisi_mean_RNA_BATCH','method']]
data_to_plot = data_to_plot[~data_to_plot.method.isin(['34_arsyn', '38_harman'])]
g = sns.pairplot(
        data=data_to_plot, #ax=ax,
        hue='method', palette=method_pal
        #palette=dict(zip(order, colors)),
        #size=strip_size, alpha=strip_alpha, jitter=True, dodge=False,
    )
#g.legend(loc='right', bbox_to_anchor=(1.5, 0.5), ncol=1)

#### The rest plots

In [ ]:
bio_asw_cols = [c for c in df_ok.columns if c.startswith("asw_bio_norm_")]
clisi_bio_cols = [c for c in df_ok.columns if c.startswith("clisi_mean_")]

for group_col in ["method", "strat", "imp"]:
    for col in bio_asw_cols[:2] + clisi_bio_cols[:2]:
        if col in df_ok.columns:
            fig = plot_multilevel_comparison(
                df_ok, col, group_col, title=f"{col} by {group_col}"
            )
            safe_name = col.replace("/", "_").replace(" ", "_")
            fig.savefig(
                FIGURES_DIR / f"b_bio_{safe_name}_{group_col}.svg", bbox_inches="tight"
            )
            fig.savefig(
                FIGURES_DIR / f"b_bio_{safe_name}_{group_col}.png",
                bbox_inches="tight",
                dpi=200,
            )
            plt.show()

In [ ]:
batch_asw = "asw_batch_norm_RNA_BATCH"
bio_panels = [
    ("asw_bio_norm_Major_group", "Major_group"),
    ("asw_bio_norm_Diagnosis_cell_type_unified", "Diagnosis_cell_type_unified"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (bio_col, bio_label) in zip(axes, bio_panels):
    if batch_asw not in df_ok.columns or bio_col not in df_ok.columns:
        ax.set_visible(False)
        continue
    for imp_name, imp_color in imp_pal.items():
        sub = df_ok[df_ok.imp == imp_name]
        ax.scatter(
            sub[batch_asw], sub[bio_col],
            color=imp_color, label=imp_name, alpha=0.75, s=40,
        )
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
    combined = df_ok[batch_asw] + df_ok[bio_col]
    for idx in combined.nlargest(3).index:
        row = df_ok.loc[idx]
        ax.annotate(row.method, (row[batch_asw], row[bio_col]),
                    fontsize=FONT_ANNOT, alpha=0.9)
    ax.set_xlabel("ASW batch norm RNA_BATCH (↑ better)", fontsize=FONT_LABEL)
    ax.set_ylabel(f"ASW bio norm {bio_label} (↑ better)", fontsize=FONT_LABEL)
    ax.set_title(f"Batch–biology trade-off ({bio_label})", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    ax.legend(title="Imputation", fontsize=FONT_LEGEND)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "b_batch_biology_tradeoff.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "b_batch_biology_tradeoff.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
cms_present = {
    b: f"cms_fraction_mixed_{b}"
    for b in ["RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL"]
    if f"cms_fraction_mixed_{b}" in df_ok.columns
}

if cms_present:
    fig, axes = plt.subplots(1, len(cms_present), figsize=(5 * len(cms_present), 5))
    if len(cms_present) == 1:
        axes = [axes]
    for ax, (batch_name, col) in zip(axes, cms_present.items()):
        bp = ax.boxplot(
            [df_ok[df_ok.method == m][col].dropna().values for m in method_order],
            labels=method_order,
            patch_artist=True,
        )
        ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8)
        ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"CMS fraction mixed\n{batch_name}", fontsize=9)
        ax.set_ylabel("CMS fraction mixed")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "b_cms_grid.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "b_cms_grid.png", bbox_inches="tight", dpi=200)
    plt.show()

    for group_col in ["method", "strat", "imp"]:
        for col in list(cms_present.values())[:2]:
            fig = plot_multilevel_comparison(
                df_ok, col, group_col, title=f"{col} by {group_col}"
            )
            safe = col.replace("/", "_")
            fig.savefig(
                FIGURES_DIR / f"b_cms_multilevel_{safe}_{group_col}.svg",
                bbox_inches="tight",
            )
            fig.savefig(
                FIGURES_DIR / f"b_cms_multilevel_{safe}_{group_col}.png",
                bbox_inches="tight",
                dpi=200,
            )
            plt.show()

### §3.3 — Group C: Embedding Metrics

In [ ]:
batch_embed_cols = ["RNA_BATCH", "PLATFORM_RNA", "RNASEQ_SOURCE", "COHORT_LABEL"]
embed_types = [("umap", "UMAP"), ("tsne", "tSNE")]

fig, axes = plt.subplots(
    2, len(batch_embed_cols), figsize=(4 * len(batch_embed_cols), 7),
    sharey=True,
)
for row_idx, (prefix, embed_name) in enumerate(embed_types):
    for col_idx, bc in enumerate(batch_embed_cols):
        ax = axes[row_idx, col_idx]
        col = f"{prefix}_centroid_disp_{bc}"
        if col not in df_ok.columns:
            ax.set_visible(False)
            continue
        vals = [df_ok[df_ok.method == m][col].mean() for m in method_order]
        ax.bar(
            range(len(method_order)), vals,
            color=[method_pal.get(m, "#AAAAAA") for m in method_order],
        )
        ax.set_xticks(range(len(method_order)))
        ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=FONT_TINY)
        ax.set_title(f"{embed_name} disp — {bc}", fontsize=FONT_SMALL)
        if col_idx == 0:
            ax.set_ylabel("Centroid dispersion (↓)", fontsize=FONT_LABEL)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "c_centroid_disp_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "c_centroid_disp_grid.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
fig, axes = plt.subplots(
    1, len(batch_embed_cols), figsize=(5 * len(batch_embed_cols), 4)
)
for ax, bc in zip(axes, batch_embed_cols):
    uc = f"umap_entropy_norm_{bc}"
    tc = f"tsne_entropy_norm_{bc}"
    if uc not in df_ok.columns or tc not in df_ok.columns:
        ax.set_visible(False)
        continue
    x_vals = df_ok[uc].dropna()
    y_vals = df_ok[tc].dropna()
    common = x_vals.index.intersection(y_vals.index)
    x_vals, y_vals = x_vals.loc[common], y_vals.loc[common]
    ax.scatter(
        x_vals, y_vals, alpha=0.6, s=20, c=df_ok.loc[common, "method"].map(method_pal)
    )
    if len(x_vals) >= 3:
        r, p = pearsonr(x_vals, y_vals)
        ax.set_title(f"{bc}\nr={r:.2f}, p={p:.3f}", fontsize=8)
    else:
        ax.set_title(bc, fontsize=8)
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.7, alpha=0.5)
    ax.set_xlabel("UMAP entropy norm")
    ax.set_ylabel("tSNE entropy norm")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "c_umap_tsne_entropy_corr.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "c_umap_tsne_entropy_corr.png", bbox_inches="tight", dpi=200)
plt.show()

### §3.4 — Group D: Distribution Metrics

In [ ]:
ks_batch_cols = ["RNA_BATCH", "PLATFORM_RNA", "COHORT_LABEL"]
ks_present = {
    b: f"ks_mean_D_{b}" for b in ks_batch_cols if f"ks_mean_D_{b}" in df_ok.columns
}
ks_frac_map = {b: f"ks_frac_sig_{b}" for b in ks_batch_cols}

fig, axes = plt.subplots(1, len(ks_present), figsize=(5 * len(ks_present), 5))
if len(ks_present) == 1:
    axes = [axes]
for ax, (batch_name, col) in zip(axes, ks_present.items()):
    frac_col = ks_frac_map.get(batch_name)
    bp = ax.boxplot(
        [df_ok[df_ok.method == m][col].dropna().values for m in method_order],
        labels=method_order,
        patch_artist=True,
    )
    for patch, m in zip(bp["boxes"], method_order):
        mean_frac = (
            df_ok[df_ok.method == m][frac_col].mean()
            if (frac_col and frac_col in df_ok.columns)
            else 0
        )
        patch.set_facecolor("#FF4444" if mean_frac > 0.5 else "#4499FF")
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=7)
    ax.set_title(f"KS mean D\n{batch_name}", fontsize=9)
    ax.set_ylabel("Mean KS D-statistic (\u2193 better)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "d_ks_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "d_ks_grid.png", bbox_inches="tight", dpi=200)
plt.show()

for group_col in ["method", "strat", "imp"]:
    if "ks_mean_D_RNA_BATCH" in df_ok.columns:
        fig = plot_multilevel_comparison(
            df_ok,
            "ks_mean_D_RNA_BATCH",
            group_col,
            title=f"KS D RNA_BATCH by {group_col}",
        )
        fig.savefig(
            FIGURES_DIR / f"d_ks_multilevel_{group_col}.svg", bbox_inches="tight"
        )
        fig.savefig(
            FIGURES_DIR / f"d_ks_multilevel_{group_col}.png",
            bbox_inches="tight",
            dpi=200,
        )
        plt.show()

In [ ]:
if (
    "ks_mean_D_RNA_BATCH" in df_ok.columns
    and "ks_cohort_within_batch_mean_D" in df_ok.columns
):
    fig, ax = plt.subplots(figsize=(8, 6))
    for method_name in method_order:
        sub = df_ok[df_ok.method == method_name]
        ax.scatter(
            sub["ks_mean_D_RNA_BATCH"],
            sub["ks_cohort_within_batch_mean_D"],
            color=method_pal.get(method_name, "#AAAAAA"),
            label=method_name, alpha=0.7, s=50,
        )
    ax.set_xlabel("KS mean D RNA_BATCH (platform-level)", fontsize=FONT_LABEL)
    ax.set_ylabel("KS cohort within-batch mean D", fontsize=FONT_LABEL)
    ax.set_title("Platform vs within-platform cohort KS effects", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "d_ks_platform_vs_cohort.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "d_ks_platform_vs_cohort.png", bbox_inches="tight", dpi=200)
    plt.show()


In [ ]:
cv_present = {
    b: f"per_gene_batch_mean_cv_{b}"
    for b in ["RNA_BATCH", "PLATFORM_RNA", "COHORT_LABEL"]
    if f"per_gene_batch_mean_cv_{b}" in df_ok.columns
}

for batch_name, col in cv_present.items():
    means = df_ok.groupby("method")[col].mean().reindex(method_order)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(
        range(len(method_order)),
        means.values,
        color=[method_pal.get(m, "#AAAAAA") for m in method_order],
    )
    ax.set_xticks(range(len(method_order)))
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Mean per-gene batch CV (\u2193 better)")
    ax.set_title(f"Per-gene batch mean CV \u2014 {batch_name}")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"d_gene_cv_{batch_name}.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / f"d_gene_cv_{batch_name}.png", bbox_inches="tight", dpi=200
    )
    plt.show()

    for group_col in ["method", "strat", "imp"]:
        fig = plot_multilevel_comparison(
            df_ok, col, group_col, title=f"{col} by {group_col}"
        )
        fig.savefig(
            FIGURES_DIR / f"d_cv_multilevel_{batch_name}_{group_col}.svg",
            bbox_inches="tight",
        )
        fig.savefig(
            FIGURES_DIR / f"d_cv_multilevel_{batch_name}_{group_col}.png",
            bbox_inches="tight",
            dpi=200,
        )
        plt.show()

### §3.5 — Group E: Data Quality

In [ ]:
si_df = df_ok.groupby(["strat", "imp"]).first()[["n_genes", "n_samples"]].reset_index()
si_df["label"] = si_df["strat"] + "\n" + si_df["imp"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, label in [
    (axes[0], "n_genes",   "Number of genes"),
    (axes[1], "n_samples", "Number of samples"),
]:
    vals = [
        df_ok[(df_ok.strat == r.strat) & (df_ok.imp == r.imp)][col].mean()
        for _, r in si_df.iterrows()
    ]
    bar_colors = [
        strat_pal.get(r.strat, "#AAAAAA") for _, r in si_df.iterrows()
    ]
    bars = ax.bar(range(len(si_df)), vals, color=bar_colors)
    ax.set_xticks(range(len(si_df)))
    ax.set_xticklabels(si_df["label"], rotation=45, ha="right", fontsize=FONT_SMALL)
    ax.set_ylabel(label, fontsize=FONT_LABEL)
    ax.set_title(f"{label} by (strat, imp)", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 5,
            f"{int(val)}", ha="center", va="bottom", fontsize=FONT_ANNOT,
        )
fig.tight_layout()
fig.savefig(FIGURES_DIR / "e_gene_sample_counts.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "e_gene_sample_counts.png", bbox_inches="tight", dpi=200)
plt.show()

for group_col in ["strat", "imp"]:
    for col in ["n_genes", "n_samples"]:
        fig = plot_multilevel_comparison(df_ok, col, group_col, title=f"{col} by {group_col}")
        fig.savefig(FIGURES_DIR / f"e_{col}_multilevel_{group_col}.svg", bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"e_{col}_multilevel_{group_col}.png", bbox_inches="tight", dpi=200)
        plt.show()


In [ ]:
bimod_col = "fraction_cohorts_bimodal"
zi_col    = "fraction_cohorts_zero_inflated_bimodal"
if bimod_col in df_ok.columns and zi_col in df_ok.columns:
    size_col = (
        df_ok["n_cohorts"] * 2 if "n_cohorts" in df_ok.columns
        else pd.Series(50, index=df_ok.index)
    )
    fig, ax = plt.subplots(figsize=(9, 6))
    for method_name in method_order:
        sub = df_ok[df_ok.method == method_name]
        ax.scatter(
            sub[bimod_col], sub[zi_col],
            s=size_col.loc[sub.index],
            alpha=0.75,
            color=method_pal.get(method_name, "#AAAAAA"),
            label=method_name,
        )
    ax.set_xlabel("Fraction cohorts bimodal (↑ good)", fontsize=FONT_LABEL)
    ax.set_ylabel("Fraction cohorts zero-inflated bimodal (↑ bad)", fontsize=FONT_LABEL)
    ax.set_title("Bimodality landscape — size=n_cohorts, color=method", fontsize=FONT_TITLE)
    ax.tick_params(labelsize=FONT_TICK)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "e_bimodality_landscape.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "e_bimodality_landscape.png", bbox_inches="tight", dpi=200)
    plt.show()

    for group_col in ["imp", "strat"]:
        for col in [bimod_col, zi_col]:
            fig = plot_multilevel_comparison(df_ok, col, group_col, title=f"{col} by {group_col}")
            safe = col.replace("/", "_")
            fig.savefig(FIGURES_DIR / f"e_bimodal_{safe}_{group_col}.svg", bbox_inches="tight")
            fig.savefig(FIGURES_DIR / f"e_bimodal_{safe}_{group_col}.png", bbox_inches="tight", dpi=200)
            plt.show()


In [ ]:
exp_stat_cols = [
    c for c in ["exp_median", "exp_std", "exp_p01", "exp_p99"] if c in df_ok.columns
]
pivot = df_ok.groupby("method")[exp_stat_cols].mean().reindex(method_order)

row_strips = pd.DataFrame(
    {
        "strat": df_ok.groupby("method")["strat"]
        .first()
        .reindex(method_order)
        .map(strat_pal)
        .fillna("#AAAAAA"),
        "imp": df_ok.groupby("method")["imp"]
        .first()
        .reindex(method_order)
        .map(imp_pal)
        .fillna("#AAAAAA"),
    },
    index=method_order,
)

g_exp = sns.clustermap(
    pivot.fillna(pivot.median()),
    row_cluster=False,
    col_cluster=False,
    row_colors=row_strips,
    cmap="RdYlBu_r",
    standard_scale=1,
    figsize=(10, 8),
    linewidths=0.3,
    annot=True,
    fmt=".2f",
)
g_exp.ax_heatmap.set_title("Expression statistics by method")
g_exp.fig.savefig(FIGURES_DIR / "e_exp_stats_heatmap.svg", bbox_inches="tight")
g_exp.fig.savefig(FIGURES_DIR / "e_exp_stats_heatmap.png", bbox_inches="tight", dpi=200)
plt.show()

### §3.6 — Groups G and H: Graph Connectivity and Euclidean Distances

In [ ]:
graph_cols = [c for c in df_ok.columns if c.startswith("graph_connectivity_")]

fig, axes = plt.subplots(1, len(graph_cols), figsize=(5 * len(graph_cols), 5), sharey=True)
if len(graph_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, graph_cols):
    bio_label = col.replace("graph_connectivity_", "")
    bp = ax.boxplot(
        [df_ok[df_ok.method == m][col].dropna().values for m in method_order],
        labels=method_order,
        patch_artist=True,
    )
    for patch, m in zip(bp["boxes"], method_order):
        patch.set_facecolor(method_pal.get(m, "#AAAAAA"))
        patch.set_alpha(0.7)
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=FONT_TINY)
    ax.set_title(f"Graph connectivity — {bio_label}", fontsize=FONT_SMALL)
    if ax == axes[0]:
        ax.set_ylabel("Graph connectivity (↑ better)", fontsize=FONT_LABEL)
    ax.tick_params(axis="y", labelsize=FONT_TICK)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "g_graph_connectivity_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "g_graph_connectivity_grid.png", bbox_inches="tight", dpi=200)
plt.show()

for group_col in ["method", "strat", "imp"]:
    for col in graph_cols[:2]:
        fig = plot_multilevel_comparison(df_ok, col, group_col, title=f"{col} by {group_col}")
        safe = col.replace("/", "_")
        fig.savefig(FIGURES_DIR / f"g_graph_multilevel_{safe}_{group_col}.svg", bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"g_graph_multilevel_{safe}_{group_col}.png", bbox_inches="tight", dpi=200)
        plt.show()


In [ ]:
dr_batch = ["dist_ratio_RNA_BATCH", "dist_ratio_PLATFORM_RNA"]
dr_bio   = ["dist_ratio_Major_group", "dist_ratio_Diagnosis_cell_type_unified"]
dr_all   = [c for c in (dr_batch + dr_bio) if c in df_ok.columns]

fig, axes = plt.subplots(1, len(dr_all), figsize=(4 * len(dr_all), 5), sharey=True)
if len(dr_all) == 1:
    axes = [axes]
for ax, col in zip(axes, dr_all):
    label = col.replace("dist_ratio_", "")
    vals  = df_ok.groupby("method")[col].mean().reindex(method_order)
    ax.bar(
        range(len(method_order)), vals.values,
        color=[method_pal.get(m, "#AAAAAA") for m in method_order],
    )
    ax.set_xticks(range(len(method_order)))
    ax.set_xticklabels(method_order, rotation=45, ha="right", fontsize=FONT_TINY)
    ax.set_title(f"Distance ratio — {label}", fontsize=FONT_SMALL)
    if ax == axes[0]:
        ax.set_ylabel("Distance ratio (↓ better)", fontsize=FONT_LABEL)
    ax.tick_params(axis="y", labelsize=FONT_TICK)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "h_dist_ratio_grid.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "h_dist_ratio_grid.png", bbox_inches="tight", dpi=200)
plt.show()


### §3.7 — Parallel coordinates for top-N methods


In [ ]:
# ── §3.7.1 Parallel coordinates — top-N methods on key metrics ───────────────
from pandas.plotting import parallel_coordinates

display_metrics = [
    c for c in [
        "r2_RNA_BATCH", "kbet_acceptance_rate_RNA_BATCH",
        "umap_entropy_norm_RNA_BATCH", "ks_mean_D_RNA_BATCH",
        "n_genes", "graph_connectivity_Diagnosis_cell_type_unified",
        "dist_ratio_Diagnosis_cell_type_unified", "composite_score",
    ] if c in df_normed.columns or c in df_ok.columns
]

top_methods = df_ok.nlargest(N_TOP, "composite_score")["method"].unique()
_pc_df = (
    df_normed.reindex(df_ok[df_ok.method.isin(top_methods)].index)
    [[c for c in display_metrics if c in df_normed.columns]]
    .assign(method=df_ok.method)
    .dropna()
)

fig, ax = plt.subplots(figsize=(14, 6))
parallel_coordinates(
    _pc_df, "method",
    color=[method_pal.get(m, "#AAAAAA") for m in _pc_df.method.unique()],
    ax=ax, linewidth=1.5, alpha=0.7,
)
ax.set_ylabel("Normalized score (1 = best)", fontsize=FONT_LABEL)
ax.tick_params(axis="x", labelsize=FONT_SMALL, rotation=30)
ax.tick_params(axis="y", labelsize=FONT_TICK)
ax.set_title("Parallel coordinates: top-N methods across key metrics", fontsize=FONT_TITLE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "parallel_coords_top_methods.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "parallel_coords_top_methods.png", bbox_inches="tight", dpi=200)
plt.show()


### §3.8 — Ranking stability heatmap


In [ ]:
# ── §3.8.1 Rank stability across metric subsets ───────────────────────────────
rank_variants: dict = {}
full_scores = df_ok[scoring_cols].fillna(0)
full_composite = df_normed[scoring_cols].mean(axis=1)
rank_variants["full"] = full_composite.rank(ascending=False)

for excluded_group in "ABCDEFGH":
    excl_cols = [c for c in scoring_cols if col_meta.loc[c, "group"] == excluded_group]
    remaining = [c for c in scoring_cols if c not in excl_cols]
    if not remaining:
        continue
    sub_score = df_normed[remaining].mean(axis=1)
    rank_variants[f"no_{excluded_group}"] = sub_score.rank(ascending=False)

rank_df = pd.DataFrame(rank_variants)
top20_idx = full_composite.nlargest(20).index
rank_df = rank_df.loc[rank_df.index.intersection(top20_idx)]
rank_labels = df_ok.loc[rank_df.index, "method"] + " / " + df_ok.loc[rank_df.index, "strat"]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    rank_df.set_index(rank_labels),
    annot=True, fmt=".0f", cmap="YlOrRd_r", ax=ax,
    annot_kws={"size": FONT_TINY},
    linewidths=0.3, linecolor="#DDDDDD",
    cbar_kws={"label": "Rank (lower = better)"},
)
ax.set_title("Ranking stability: removing each metric group", fontsize=FONT_TITLE)
ax.tick_params(axis="x", labelsize=FONT_SMALL)
ax.tick_params(axis="y", labelsize=FONT_SMALL)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "rank_stability_heatmap.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "rank_stability_heatmap.png", bbox_inches="tight", dpi=200)
plt.show()


### §3.9 — Bubble chart: method × metric group summary


In [ ]:
# ── §3.9.1 Bubble chart: method × metric group ────────────────────────────────
method_order_comp = (
    df_ok.groupby("method")["composite_score"].mean()
    .sort_values(ascending=False)
    .index.tolist()
)
bubble_df = pd.DataFrame({
    grp: df_normed.assign(method=df_ok.method)
         .groupby("method")[[c for c in scoring_cols if col_meta.loc[c, "group"] == grp]]
         .mean().mean(axis=1)
    for grp in "ABCDEFGH"
    if any(col_meta.loc[c, "group"] == grp for c in scoring_cols)
})

fig, ax = plt.subplots(figsize=(12, 9))
for col_idx, grp in enumerate(bubble_df.columns):
    for row_idx, method_name in enumerate(method_order_comp):
        if method_name not in bubble_df.index:
            continue
        score = bubble_df.loc[method_name, grp]
        ax.scatter(col_idx, -row_idx,
                   s=max(score * 500, 5),
                   c=group_pal.get(grp, "#AAAAAA"),
                   alpha=0.8, linewidths=0)

ax.set_xticks(range(len(bubble_df.columns)))
ax.set_xticklabels([f"Group {g}" for g in bubble_df.columns], fontsize=FONT_LABEL)
ax.set_yticks(-np.arange(len(method_order_comp)))
ax.set_yticklabels(method_order_comp, fontsize=FONT_SMALL)
ax.set_title("Normalized score by method × metric group (bubble size = mean score)",
             fontsize=FONT_TITLE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bubble_method_x_group.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "bubble_method_x_group.png", bbox_inches="tight", dpi=200)
plt.show()


---
## §4 — Composite Scoring and Top-N Selection

### §4.1 — Literature review: metric weighting approaches

**Batch mixing metrics (60% weight):**
PCA R\u00b2 is the most widely reported metric in harmonization benchmarks.
Tran et al. 2020 (*Nature Methods*) used it as the primary outcome in a benchmark of 14 single-cell integration methods.
Luecken et al. 2022 (scIB benchmark, *Nature Methods*) report kBET (B\u00fcttner et al. 2019) as the primary and
ASW\_batch as the secondary batch correction score.
The LISI metric (Korsunsky et al. 2019, Harmony paper) is complementary \u2014 it operates in reduced-dimension space
and measures local neighborhood mixing.
The scIB benchmark assigns 60% of the overall score to batch correction, motivating the 0.60 component weight here.

**Biology preservation metrics (35% weight):**
ASW\_bio and graph connectivity are the standard biology preservation metrics in scIB (Luecken et al. 2022).
R\u00b2 of biology covariates (Major\_group, Diagnosis\_cell\_type\_unified) on PCA axes should remain high
after batch correction \u2014 if biology R\u00b2 collapses, the correction is over-aggressive.
The 40% biology weight in scIB directly motivates the 0.35 component weight here
(slightly down-weighted because batch correction is the primary objective).

**Data quality metrics (5% weight):**
Gene retention and bimodality are upstream properties of data preparation (not the normalization itself)
and are rarely used in published benchmarks.
The 0.05 weight reflects their role as a sanity check.
Bimodality coefficient is **positive** because bimodal gene expression distributions are biologically natural
(reflecting two cell-state populations) and methods that preserve this structure are preferred.

In [ ]:
# ── Weight rationale ──────────────────────────────────────────────────────────
# All weights are positive because df_normed already applies polarity (1=best for all).
# Weight magnitude encodes relative importance within each component.
#
# Component split (60 / 35 / 5):
#   Mirrors the scIB benchmark (Luecken et al. 2022, Nature Methods) which assigns
#   60% to batch correction and 40% to biology preservation. We use 35/5 for biology
#   vs. data quality because gene count is upstream of normalization (not a correction
#   metric) and bimodality is a sanity check rather than a primary outcome.
#
# Within batch_mixing:
#   r2_RNA_BATCH weight=3.0: primary metric in virtually all published benchmarks;
#     platform-level PCA variance is the most interpretable summary.
#   pcr_RNA_BATCH weight=2.0: variance-weighted version; more sensitive to PCs 2–10.
#   kbet, ilisi, asw_batch weight=1.5: three independent neighbor-graph metrics;
#     collectively cover local mixing at different scales.
#   cms, umap_entropy, dsc, ks weight=1.0: supporting metrics with lower reliability
#     or narrower scope.
#   PLATFORM_RNA, RNASEQ_SOURCE variants weight=0.5: secondary batch groupings;
#     correction at RNA_BATCH level subsumes platform-level correction.
#
# Within bio_preservation:
#   Diagnosis_cell_type_unified metrics > Major_group because Diagnosis is the
#   fine-grained biology relevant to FL subtyping (C1/C2/C3 × GCB/ABC).
#   r2_TUMOR_NORMAL weight=1.5: tumor vs normal is a fundamental biology axis
#     that must be preserved; loss here = complete analysis failure.
#   clisi entries are polarity-corrected by POLARITY dict (1=better); weight
#   magnitude matches the corresponding ASW entry.
#
# Within data_quality:
#   n_genes weight=1.0: log-normalized (see compute_composite_score) so
#     differences in the 3,000–25,000 range matter more than raw count.
#   fraction_cohorts_bimodal weight=1.0: bimodal distributions are expected
#     in gene expression and indicate the normalization does not over-smooth.
#   per_gene_batch_mean_cv weight=0.5: secondary check; already captured by
#     ks and r2 metrics.
SCORE_WEIGHTS = {
    "batch_mixing": {
        "r2_RNA_BATCH": 3.0,
        "pcr_RNA_BATCH": 2.0,
        "kbet_acceptance_rate_RNA_BATCH": 1.5,
        "ilisi_norm_RNA_BATCH": 1.5,
        "asw_batch_norm_RNA_BATCH": 1.5,
        "cms_fraction_mixed_RNA_BATCH": 1.0,
        "umap_entropy_norm_RNA_BATCH": 1.0,
        "dsc_RNA_BATCH": 1.0,
        "ks_mean_D_RNA_BATCH": 1.0,
        "r2_PLATFORM_RNA": 0.5,
        "r2_RNASEQ_SOURCE": 0.5,
    },
    "bio_preservation": {
        "r2_Major_group": 2.0,
        "r2_TUMOR_NORMAL": 2.0,
        "asw_bio_norm_Diagnosis_cell_type_unified": 1.5,
        "asw_bio_norm_Major_group": 1.0,
        "clisi_mean_Diagnosis_cell_type_unified": 1.5,
        "clisi_mean_Major_group": 0.5,
        "graph_connectivity_Diagnosis_cell_type_unified": 1.0,
        "graph_connectivity_Major_group": 0.5,
        "dist_ratio_Diagnosis_cell_type_unified": 1.0,
        "dist_ratio_Major_group": 0.5,
    },
    "data_quality": {
        "n_genes": 0.5,
        "fraction_cohorts_bimodal": 1.0,
        "per_gene_batch_mean_cv_RNA_BATCH": 0.5,
    },
}

COMPONENT_WEIGHTS = {
    "batch_mixing": 0.60,
    "bio_preservation": 0.35,
    "data_quality": 0.05,
}


def compute_composite_score(df_normed_in, score_weights, component_weights):
    """
    Compute a weighted composite harmonization score from normalized metrics.

    Aggregates normalized metric columns (all in [0, 1] where 1=best) into
    component scores (batch_mixing, bio_preservation, data_quality), then
    combines them using component-level weights.

    Parameters
    ----------
    df_normed_in : pd.DataFrame
        Normalized metrics DataFrame (output of ``normalize_for_display``).
        Index = run IDs; columns include all metric columns.
    score_weights : dict[str, dict[str, float]]
        Nested dict: component name → {metric_col: weight}.
        Metrics absent from ``df_normed_in`` are silently skipped.
    component_weights : dict[str, float]
        Weight for each component in the final composite.  Need not sum to 1;
        weights are applied as-is (unnormalized).

    Returns
    -------
    component_df : pd.DataFrame
        Per-run component scores (one column per component key).
    composite : pd.Series
        Final composite score per run (weighted sum of components).
    """
    """
    Compute a weighted composite harmonization score from normalized metrics.

    Aggregates normalized metric columns (all in [0, 1] where 1=best) into
    component scores (batch_mixing, bio_preservation, data_quality), then
    combines them using component-level weights.

    Parameters
    ----------
    df_normed_in : pd.DataFrame
        Normalized metrics DataFrame (output of ``normalize_for_display``).
        Index = run IDs; columns include all metric columns.
    score_weights : dict[str, dict[str, float]]
        Nested dict: component name → {metric_col: weight}.
        Metrics absent from ``df_normed_in`` are silently skipped.
    component_weights : dict[str, float]
        Weight for each component in the final composite.  Need not sum to 1;
        weights are applied as-is (unnormalized).

    Returns
    -------
    component_df : pd.DataFrame
        Per-run component scores (one column per component key).
    composite : pd.Series
        Final composite score per run (weighted sum of components).
    """
    scores = {}
    for comp, metrics_dict in score_weights.items():
        present = {c: w for c, w in metrics_dict.items() if c in df_normed_in.columns}
        total_w = sum(present.values()) or 1.0
        comp_score = sum(df_normed_in[c] * w / total_w for c, w in present.items())
        scores[comp] = comp_score
    final = sum(
        component_weights[k] * scores[k] for k in component_weights if k in scores
    )
    return pd.DataFrame(scores), final


component_df, composite_score = compute_composite_score(
    df_normed, SCORE_WEIGHTS, COMPONENT_WEIGHTS
)
df_ok = df_ok.copy()
df_ok["composite_score"] = composite_score
df_ok["score_batch_mixing"] = component_df["batch_mixing"]
df_ok["score_bio_preservation"] = component_df["bio_preservation"]
df_ok["score_data_quality"] = component_df.get(
    "data_quality", pd.Series(0.0, index=df_ok.index)
)

print("Top 10 by composite score:")
print(
    df_ok.nlargest(10, "composite_score")[
        [
            "strat",
            "imp",
            "method",
            "composite_score",
            "score_batch_mixing",
            "score_bio_preservation",
            "r2_RNA_BATCH",
            "r2_Major_group",
        ]
    ].to_string()
)

In [ ]:
for c in scoring_cols:
    asc = POLARITY[c] == -1
    df_ok[f"rank_{c}"] = df_ok[c].rank(ascending=asc, na_option="bottom")

rank_cols = [f"rank_{c}" for c in scoring_cols if f"rank_{c}" in df_ok.columns]
df_ok["borda_score"] = df_ok[rank_cols].mean(axis=1)
print("Top 10 by Borda count (lower rank sum = better):")
print(
    df_ok.nsmallest(10, "borda_score")[
        ["strat", "imp", "method", "borda_score", "composite_score"]
    ].to_string()
)

In [ ]:
# === USER PARAMETERS ===
N_TOP = 10
SCORE_METHOD = "composite"  # "composite" | "borda" | "batch_only" | "bio_only"
# =======================

sort_map = {
    "composite": ("composite_score", False),
    "borda": ("borda_score", True),
    "batch_only": ("score_batch_mixing", False),
    "bio_only": ("score_bio_preservation", False),
}
sort_col, ascending = sort_map[SCORE_METHOD]
top_n = (
    df_ok.nsmallest(N_TOP, sort_col) if ascending else df_ok.nlargest(N_TOP, sort_col)
)

display_cols = [
    "strat",
    "imp",
    "method",
    "composite_score",
    "score_batch_mixing",
    "score_bio_preservation",
    "r2_RNA_BATCH",
    "r2_Major_group",
]
print(f"Top {N_TOP} by {SCORE_METHOD}:")
print(top_n[display_cols].to_string())

In [ ]:
rank_display = df_ok.sort_values("composite_score", ascending=False)[
    [
        "strat",
        "imp",
        "method",
        "composite_score",
        "score_batch_mixing",
        "score_bio_preservation",
        "r2_RNA_BATCH",
        "r2_Major_group",
        "kbet_acceptance_rate_RNA_BATCH",
        "n_genes",
    ]
].copy()
rank_display.insert(0, "Rank", range(1, len(rank_display) + 1))

kbet_col_name = "kbet_acceptance_rate_RNA_BATCH"
fmt_dict = {
    "composite_score": "{:.3f}",
    "score_batch_mixing": "{:.3f}",
    "score_bio_preservation": "{:.3f}",
    "r2_RNA_BATCH": "{:.3f}",
    "r2_Major_group": "{:.3f}",
    kbet_col_name: "{:.3f}",
    "n_genes": "{:.0f}",
}

styled = (
    rank_display.style.background_gradient(subset=["composite_score"], cmap="RdYlGn")
    .background_gradient(subset=["score_batch_mixing"], cmap="RdYlGn")
    .background_gradient(subset=["score_bio_preservation"], cmap="RdYlGn")
    .background_gradient(subset=["r2_RNA_BATCH"], cmap="RdYlGn_r")
    .format(fmt_dict)
    .set_caption("All harmonization attempts ranked by composite score")
)
styled

In [ ]:
def _safe_get(row, col, default=0.0):
    """
    NaN-safe scalar accessor for radar chart value extraction.

    Radar chart axes require float values in [0, 1]; pd.NA or np.nan would cause
    matplotlib to skip the polygon vertex, producing a degenerate shape. This
    helper returns ``default`` whenever the value is missing or not a number.

    Parameters
    ----------
    row : pd.Series
        One row of the metrics DataFrame.
    col : str
        Column name to access.
    default : float
        Value to return when the column is absent or NaN (default 0.0).

    Returns
    -------
    float
    """
    val = row.get(col, default)
    return float(val) if pd.notna(val) else default


radar_metrics = {
    "Batch mixing\n(R\u00b2 inv)": lambda r: 1 - _safe_get(r, "r2_RNA_BATCH"),
    "kBET": lambda r: _safe_get(r, "kbet_acceptance_rate_RNA_BATCH"),
    "KS align\n(inv)": lambda r: 1 - _safe_get(r, "ks_mean_D_RNA_BATCH"),
    "Bio preserv\n(ASW)": lambda r: _safe_get(r, "asw_bio_norm_Major_group"),
    "Graph\nconnect": lambda r: _safe_get(r, "graph_connectivity_Major_group"),
    "Data quality": lambda r: _safe_get(r, "score_data_quality"),
}

top5 = df_ok.nlargest(5, "composite_score")
labels = list(radar_metrics.keys())
N_axes = len(labels)
angles = np.linspace(0, 2 * np.pi, N_axes, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
colors = sns.color_palette("husl", 5)

for i, (run_id, row) in enumerate(top5.iterrows()):
    vals = [max(0.0, min(1.0, fn(row))) for fn in radar_metrics.values()]
    vals += vals[:1]
    ax.plot(
        angles,
        vals,
        "o-",
        linewidth=1.5,
        color=colors[i],
        label=f"{row.method} ({row.strat}/{row.imp})",
    )
    ax.fill(angles, vals, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=9)
ax.set_ylim(0, 1)
ax.set_title("Radar chart \u2014 Top 5 methods", size=11, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=7)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "radar_top5.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "radar_top5.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
def is_pareto(x_vals, y_vals):
    """
    Identify Pareto-optimal points in a 2-D objective space.

    A point i is Pareto-optimal (non-dominated) if no other point j simultaneously
    achieves x_vals[j] >= x_vals[i] AND y_vals[j] >= y_vals[i] with at least one
    strict inequality.  Both objectives are assumed to be "higher is better".

    Parameters
    ----------
    x_vals : array-like of float
        Values of the first objective (e.g. batch mixing score).
    y_vals : array-like of float
        Values of the second objective (e.g. biology preservation score).

    Returns
    -------
    np.ndarray of bool, shape (n,)
        True at index i if point i is Pareto-optimal.
    """
    n = len(x_vals)
    pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i != j and x_vals[j] >= x_vals[i] and y_vals[j] >= y_vals[i]:
                if x_vals[j] > x_vals[i] or y_vals[j] > y_vals[i]:
                    pareto[i] = False
                    break
    return pareto


x_arr = df_ok["score_batch_mixing"].values
y_arr = df_ok["score_bio_preservation"].values
m_arr = df_ok["method"].values
pareto_mask = is_pareto(x_arr, y_arr)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    x_arr[~pareto_mask],
    y_arr[~pareto_mask],
    c=[method_pal.get(m, "#AAAAAA") for m in m_arr[~pareto_mask]],
    alpha=0.45,
    s=40,
)
ax.scatter(
    x_arr[pareto_mask],
    y_arr[pareto_mask],
    c=[method_pal.get(m, "#AAAAAA") for m in m_arr[pareto_mask]],
    alpha=1.0,
    s=120,
    edgecolors="black",
    linewidths=1.5,
    zorder=5,
)
for xi, yi, mi in zip(x_arr[pareto_mask], y_arr[pareto_mask], m_arr[pareto_mask]):
    ax.annotate(mi, (xi, yi), fontsize=8, xytext=(5, 3), textcoords="offset points")

ax.set_xlabel("Score \u2014 batch mixing (\u2191 better)")
ax.set_ylabel("Score \u2014 biology preservation (\u2191 better)")
ax.set_title(
    "Pareto frontier: batch correction vs biology preservation\n(larger markers = Pareto-optimal)"
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pareto_frontier.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "pareto_frontier.png", bbox_inches="tight", dpi=200)
plt.show()
print(f"Pareto-optimal methods: {sorted(set(m_arr[pareto_mask]))}")

### §4.9 — Rank correlation between key metrics


In [ ]:
# ── §4.9.1 Spearman rank correlation — top scoring metrics ───────────────────
key_metrics = [c for c in SCORE_WEIGHTS if c in scoring_cols] if "SCORE_WEIGHTS" in dir() else scoring_cols[:20]
top_metric_corr = df_ok[key_metrics].corr(method="spearman")

g_km = sns.clustermap(
    top_metric_corr,
    method="ward", metric="euclidean",
    row_colors=metric_colors.reindex(key_metrics).fillna("#AAAAAA"),
    col_colors=metric_colors.reindex(key_metrics).fillna("#AAAAAA"),
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.3, figsize=(16, 14),
    xticklabels=True, yticklabels=True,
    annot_kws={"size": FONT_TINY},
)
g_km.fig.suptitle("Rank correlation between top scoring metrics",
                  fontsize=FONT_TITLE, y=1.01)
g_km.fig.savefig(FIGURES_DIR / "key_metric_rank_corr.svg", bbox_inches="tight")
g_km.fig.savefig(FIGURES_DIR / "key_metric_rank_corr.png", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
# ── §4.3 Radar chart: top-N methods on key metrics ───────────────────────────
# Axes = representative metrics from each group A–H.
# Each polygon = one method from top_n, colored by method_pal.

_RADAR_METRICS = [
    m
    for m in [
        "r2_RNA_BATCH",
        "pcr_RNA_BATCH",
        "kbet_acceptance_rate_RNA_BATCH",
        "ilisi_norm_RNA_BATCH",
        "asw_batch_norm_RNA_BATCH",
        "asw_bio_norm_Major_group",
        "umap_centroid_disp_RNA_BATCH",
        "ks_mean_D_RNA_BATCH",
        "n_genes",
        "fraction_cohorts_bimodal",
        "graph_connectivity_Major_group",
        "dist_ratio_RNA_BATCH",
    ]
    if m in df_normed.columns
]

N_AXES = len(_RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N_AXES, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(
    [m.replace("_RNA_BATCH", "").replace("_Major_group", "") for m in _RADAR_METRICS],
    size=8,
)
ax.set_ylim(0, 1)

for _, row in top_n.iterrows():
    vals = [
        _safe_get(df_normed.loc[row.name] if row.name in df_normed.index else row, m)
        for m in _RADAR_METRICS
    ]
    vals += vals[:1]
    color = method_pal.get(row["method"], "#AAAAAA")
    ax.plot(angles, vals, linewidth=1.5, color=color, alpha=0.8)
    ax.fill(angles, vals, alpha=0.08, color=color)

handles = [
    mpatches.Patch(
        color=method_pal.get(r["method"], "#AAAAAA"),
        label=f'{r["method"]} ({r["composite_score"]:.3f})',
    )
    for _, r in top_n.iterrows()
]
ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.4, 1.15), fontsize=7)
ax.set_title(f"Top-{N_TOP} methods — radar chart (normalized scores, 1=best)", pad=20)
fig.savefig(FIGURES_DIR / "score_radar_top_n.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "score_radar_top_n.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# ── §4.4 Pareto frontier: batch mixing vs biology preservation ────────────────
x = df_ok["score_batch_mixing"].values
y = df_ok["score_bio_preservation"].values
pareto_mask = is_pareto(x, y)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    x[~pareto_mask],
    y[~pareto_mask],
    alpha=0.4,
    s=20,
    color="#AAAAAA",
    label="dominated",
)
ax.scatter(
    x[pareto_mask],
    y[pareto_mask],
    alpha=0.9,
    s=50,
    color="#d62728",
    label="Pareto-optimal",
    zorder=5,
)

top_mask = df_ok.index.isin(top_n.index)
ax.scatter(
    x[top_mask],
    y[top_mask],
    alpha=1.0,
    s=80,
    color="#1f77b4",
    marker="*",
    label="top-N",
    zorder=6,
)

pareto_sorted = np.argsort(x[pareto_mask])
ax.plot(
    x[pareto_mask][pareto_sorted],
    y[pareto_mask][pareto_sorted],
    color="#d62728",
    linewidth=1,
    linestyle="--",
    alpha=0.6,
)

for idx in np.where(pareto_mask)[0]:
    ax.annotate(
        df_ok.iloc[idx]["method"],
        (x[idx], y[idx]),
        fontsize=5,
        alpha=0.7,
        xytext=(2, 2),
        textcoords="offset points",
    )

ax.set_xlabel("Batch mixing score (higher=better)")
ax.set_ylabel("Biology preservation score (higher=better)")
ax.set_title("Pareto frontier: batch mixing vs biology preservation")
ax.legend(fontsize=9)
sns.despine(ax=ax)
fig.savefig(FIGURES_DIR / "score_pareto_frontier.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "score_pareto_frontier.png", bbox_inches="tight", dpi=200)
plt.show()

---
## §5 — Visual Inspection of Top Approaches - in a Separate Notebook

Download harmonized expression matrices from S3 for the top-N methods and the raw baseline, then compute PCA/UMAP/tSNE embeddings to visually verify that batch mixing and biology separation match what the metrics indicate.

# Part 2 - Gene Sets Analysis

## Gene Set Sources and Citations

| Gene set | Source | DOI / URL |
|---|---|---|
| FL transcriptomic subtypes C1/C2/C3 (FL_2025_MARKERS) | Xochelli et al. 2025, *Leukemia* | https://doi.org/10.1038/s41375-025-02603-9 |
| legacy internal GC B-cell markers (Centroblast/Centrocyte) | unpublished internal panel | superseded by the Dybkaer et al. 2015 panels |
| FL PFS prognostic signature (FL_PROG_SIGNATURES) | Pastore et al. 2019, *J Clin Oncol* | https://doi.org/10.1200/JCO.18.01545 (PMID 29475724) |
| POD24 mutation markers | Huet et al. 2018, *Blood* | https://doi.org/10.1182/blood-2018-03-837443 |
| PROGENy pathway activating genes | Schubert et al. 2018, *Nature Communications* | https://doi.org/10.1038/s41467-017-02391-6 |
| Housekeeping genes (stability reference) | Eisenberg & Levanon 2013, *Trends Genet* | https://doi.org/10.1016/j.tig.2013.05.010 |
| kBET batch metric | Büttner et al. 2019, *Nature Methods* | https://doi.org/10.1038/s41592-018-0254-1 |
| LISI / Harmony metric | Korsunsky et al. 2019, *Nature Methods* | https://doi.org/10.1038/s41592-019-0619-0 |
| scIB benchmark + ASW/kBET weighting | Luecken et al. 2022, *Nature Methods* | https://doi.org/10.1038/s41592-021-01336-8 |
| FSQN normalization | Franks et al. 2018, *Biostatistics* | https://doi.org/10.1093/biostatistics/kxx053 |

In [ ]:
# ── §5.6 FL/GC marker gene correlation heatmap + housekeeping gene stability ──

_HK_GENES = [
    "ACTB", "GAPDH", "B2M", "HMBS", "HPRT1", "PPIA", "RPL13A",
    "RPLP0", "TBP", "YWHAZ", "UBC", "VIM", "LDHA", "PGK1", "SDHA",
]

_fl_genes_for_corr: list[str] = []

try:
    _method_for_corr = top_n.iloc[0]
    exp_corr, _ = load_and_cache(
        _method_for_corr.strat, _method_for_corr.imp, _method_for_corr.method
    )
    if _fl_genes_for_corr:
        _genes_avail = [g for g in _fl_genes_for_corr if g in exp_corr.columns]
        if len(_genes_avail) >= 5:
            corr_mat = exp_corr[_genes_avail].corr(method="pearson")
            fig_corr = sns.clustermap(
                corr_mat, cmap="coolwarm", vmin=-1, vmax=1,
                figsize=(12, 12), linewidths=0.3,
            )
            fig_corr.fig.suptitle(
                f"FL marker gene Pearson correlation — {_method_for_corr.method}",
                fontsize=FONT_TITLE,
            )
            fig_corr.savefig(FIGURES_DIR / "fl_marker_gene_corr.svg", bbox_inches="tight")
            fig_corr.savefig(FIGURES_DIR / "fl_marker_gene_corr.png", bbox_inches="tight", dpi=200)
            plt.show()
        else:
            print(f"Too few FL genes in expression matrix ({len(_genes_avail)}); skipping Panel A.")
    else:
        print("FL_GENES_ALL not yet defined — run §11 first, then re-run this cell.")
except Exception as exc:
    print(f"Panel A skipped: {exc}")

_hk_rows = []
for strat, imp, method in _top5_ids:
    try:
        exp_hk, ann_hk = load_and_cache(strat, imp, method)
        hk_avail = [g for g in _HK_GENES if g in exp_hk.columns]
        for g in hk_avail:
            vals      = exp_hk[g]
            batch_cvs = ann_hk.groupby("RNA_BATCH").apply(
                lambda grp: vals.loc[grp.index].std() / (vals.loc[grp.index].mean() or np.nan)
            )
            bio_cvs   = ann_hk.groupby("Major_group").apply(
                lambda grp: vals.loc[grp.index].std() / (vals.loc[grp.index].mean() or np.nan)
            )
            _hk_rows.append({
                "method": method, "gene": g,
                "batch_cv": batch_cvs.mean(), "bio_cv": bio_cvs.mean(),
            })
    except Exception:
        pass

if _hk_rows:
    hk_df          = pd.DataFrame(_hk_rows)
    hk_pivot_batch = hk_df.pivot(index="gene", columns="method", values="batch_cv")
    hk_pivot_bio   = hk_df.pivot(index="gene", columns="method", values="bio_cv")

    fig_hk, (ax_b, ax_bio) = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(hk_pivot_batch, ax=ax_b, cmap="YlOrRd",
                cbar_kws={"label": "Mean CV across RNA_BATCH"},
                annot_kws={"size": FONT_TINY})
    ax_b.set_title("Housekeeping gene CV — batch axis (lower=better)", fontsize=FONT_TITLE)
    ax_b.tick_params(labelsize=FONT_TICK)
    sns.heatmap(hk_pivot_bio, ax=ax_bio, cmap="YlOrRd",
                cbar_kws={"label": "Mean CV across Major_group"},
                annot_kws={"size": FONT_TINY})
    ax_bio.set_title("Housekeeping gene CV — biology axis", fontsize=FONT_TITLE)
    ax_bio.tick_params(labelsize=FONT_TICK)
    fig_hk.tight_layout()
    fig_hk.savefig(FIGURES_DIR / "fl_hk_gene_cv.svg", bbox_inches="tight")
    fig_hk.savefig(FIGURES_DIR / "fl_hk_gene_cv.png", bbox_inches="tight", dpi=200)
    plt.show()
else:
    print("Housekeeping CV data unavailable — check S3 access.")


### §5.7 — Ridgeline plots: expression distribution per RNA_BATCH


In [ ]:
# ── §5.7.1 Ridgeline (joy) plots: stacked density curves per RNA_BATCH ───────
from scipy.stats import gaussian_kde

methods_to_plot = [
    t for t in [
        ("A_confirmed_bad", "strict", "16_fsqn_r"),
        ("A_confirmed_bad", "strict", "17_quantile"),
        ("A_confirmed_bad", "strict", "01_raw"),
    ]
    if any(df_ok.method.eq(t[2]) & df_ok.strat.eq(t[0]) & df_ok.imp.eq(t[1]))
]

if methods_to_plot:
    fig, axes = plt.subplots(1, len(methods_to_plot),
                              figsize=(6 * len(methods_to_plot), 8))
    if len(methods_to_plot) == 1:
        axes = [axes]
    for ax, (strat, imp, method) in zip(axes, methods_to_plot):
        try:
            exp, ann = load_and_cache(strat, imp, method)
            batches  = ann["RNA_BATCH"].dropna().unique()
            flat     = exp.values.ravel()
            x_range  = np.linspace(np.nanpercentile(flat, 1), np.nanpercentile(flat, 99), 300)
            for i, batch in enumerate(sorted(batches)):
                sub = exp.loc[ann.RNA_BATCH == batch].values.ravel()
                sub = sub[np.isfinite(sub)]
                if len(sub) < 20:
                    continue
                kde    = gaussian_kde(sub, bw_method=0.1)
                y      = kde(x_range)
                offset = -i * (y.max() * 0.8)
                color  = rna_batch_palette.get(batch, "#AAAAAA")
                ax.fill_between(x_range, offset, offset + y, color=color, alpha=0.6)
                ax.plot(x_range, offset + y, color="white", linewidth=0.5)
                ax.text(x_range[-1], offset + y.max() * 0.5, batch,
                        fontsize=FONT_TINY, ha="left", va="center")
            ax.set_title(method, fontsize=FONT_TITLE)
            ax.set_xlabel("Expression (log2)", fontsize=FONT_LABEL)
            ax.set_yticks([])
        except Exception as exc:
            ax.text(0.5, 0.5, str(exc), ha="center", va="center",
                    transform=ax.transAxes, fontsize=FONT_SMALL)
    fig.suptitle("Expression distribution ridgelines per RNA_BATCH", fontsize=FONT_TITLE + 2)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "ridgeline_rna_batch.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "ridgeline_rna_batch.png", bbox_inches="tight", dpi=200)
    plt.show()
else:
    print("No matching methods found in df_ok for ridgeline plot.")


---
## §6 — Download Gene Lists from S3

Download per-harmonization gene list JSON files and build a structured dictionary for gene-level analysis (§7–§11).

In [ ]:
import botocore.exceptions


def list_gene_files(s3_bucket: str, s3_prefix: str) -> list[str]:
    """
    List all per-harmonization gene list JSON files on S3.

    Parameters
    ----------
    s3_bucket : str
        S3 bucket name.
    s3_prefix : str
        S3 path prefix.

    Returns
    -------
    list of str
        Full S3 keys of all gene list files found.
    """
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=s3_bucket, Prefix=f"{s3_prefix}/genes/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("_genes.json"):
                keys.append(obj["Key"])
    return keys


try:
    gene_file_keys = list_gene_files(S3_BUCKET, S3_PREFIX)
    print(f"Gene list files found: {len(gene_file_keys)}")
except botocore.exceptions.ClientError as exc:
    gene_file_keys = []
    print(f"S3 access error: {exc}")

In [ ]:
def parse_gene_key(s3_key: str) -> tuple[str, str, str, bool] | None:
    """
    Parse a gene list S3 key into its (strat, imp, method, post_rm) components.

    Parameters
    ----------
    s3_key : str
        Full S3 key string.

    Returns
    -------
    tuple (strat, imp, method, post_rm) or None if the key cannot be parsed.
    """
    fname = s3_key.split("/")[-1].replace("_genes.json", "")
    parts = fname.split("__")
    if len(parts) != 4:
        return None
    strat, imp, method, pm_tag = parts
    return strat, imp, method, (pm_tag == "post1")


gene_sets: dict[tuple[str, str, str, bool], set[str]] = {}
_s3_client = boto3.client("s3")

for key in gene_file_keys:
    parsed = parse_gene_key(key)
    if parsed is None:
        continue
    try:
        body = _s3_client.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()
        genes = set(json.loads(body))
        gene_sets[parsed] = genes
    except botocore.exceptions.ClientError:
        pass

print(f"Gene sets loaded: {len(gene_sets)}")

In [ ]:
gene_df = pd.DataFrame(
    [
        {"strat": k[0], "imp": k[1], "method": k[2], "post_rm": k[3], "n_genes": len(v)}
        for k, v in gene_sets.items()
    ]
)

if len(gene_df):
    print(gene_df.groupby(["strat", "imp", "post_rm"])["n_genes"].describe())
    print("\nGene count by method:")
    print(gene_df.groupby("method")["n_genes"].mean().sort_values())
else:
    print("No gene sets loaded — check S3 access.")

In [ ]:
# Aggregate gene sets by (strat, imp) — union across all methods except 25_angel.
# 25_angel applies its own internal gene filtering on top of the preparation step
# and is tracked separately.
gene_sets_by_strat_imp: dict[tuple[str, str], set[str]] = {}
gene_sets_angel: dict[tuple[str, str], set[str]] = {}

for (strat, imp, method, post_rm), genes in gene_sets.items():
    if not post_rm:
        key = (strat, imp)
        if method == "25_angel":
            gene_sets_angel.setdefault(key, set()).update(genes)
        else:
            gene_sets_by_strat_imp.setdefault(key, set()).update(genes)

print(f"(strat, imp) pairs with gene sets: {len(gene_sets_by_strat_imp)}")
print(f"Angel-specific gene sets: {len(gene_sets_angel)}")

---
## §7 — Gene Count and Overlap Analysis

How does the number of genes available for analysis change as sample removal strategy becomes more aggressive? What is the pairwise overlap between strategies and imputation methods?

In [ ]:
# ── §7.1 Gene count per (strat, imp) ─────────────────────────────────────────

if gene_sets_by_strat_imp:
    _gene_counts = pd.DataFrame(
        [
            {"strat": s, "imp": i, "n_genes": len(g)}
            for (s, i), g in gene_sets_by_strat_imp.items()
        ]
    )
    _gene_counts_a = pd.DataFrame(
        [
            {"strat": s, "imp": i, "n_genes": len(g), "method": "25_angel"}
            for (s, i), g in gene_sets_angel.items()
        ]
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    _ordered = [
        (s, i)
        for s in ALL_STRATS
        for i in ["strict", "knn", "softimpute"]
        if (s, i) in gene_sets_by_strat_imp
    ]
    _labels = [f"{s}\n{i}" for s, i in _ordered]
    _counts = [len(gene_sets_by_strat_imp[(s, i)]) for s, i in _ordered]
    _colors = [strat_pal.get(s, "#AAAAAA") for s, i in _ordered]
    ax1.bar(range(len(_ordered)), _counts, color=_colors, edgecolor="white")
    ax1.set_xticks(range(len(_ordered)))
    ax1.set_xticklabels(_labels, rotation=90, fontsize=7)
    ax1.set_ylabel("Number of genes")
    ax1.set_title("Gene count per (strat, imp) — all methods union")

    if len(_gene_counts_a):
        _a_ordered = [s for s in ALL_STRATS if (s, "strict") in gene_sets_angel]
        _a_counts = [len(gene_sets_angel.get((s, "strict"), set())) for s in _a_ordered]
        ax2.bar(
            range(len(_a_ordered)),
            _a_counts,
            color=[strat_pal.get(s, "#AAAAAA") for s in _a_ordered],
        )
        ax2.set_xticks(range(len(_a_ordered)))
        ax2.set_xticklabels(_a_ordered, rotation=45, ha="right", fontsize=8)
        ax2.set_ylabel("Number of genes")
        ax2.set_title("Gene count for 25_angel (strict imp)")

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "gene_count_by_strat_imp.svg", bbox_inches="tight")
    fig.savefig(
        FIGURES_DIR / "gene_count_by_strat_imp.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("gene_sets_by_strat_imp is empty — run §6 first.")

In [ ]:
# ── §7.2 Pairwise Jaccard overlap heatmap ─────────────────────────────────────


def jaccard_matrix(sets: list[set]) -> np.ndarray:
    """
    Compute the pairwise Jaccard similarity matrix for a list of gene sets.

    Parameters
    ----------
    sets : list of set

    Returns
    -------
    np.ndarray, shape (n, n) — symmetric Jaccard matrix, values in [0, 1].
    """
    n = len(sets)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            inter = len(sets[i] & sets[j])
            union = len(sets[i] | sets[j])
            mat[i, j] = mat[j, i] = inter / union if union > 0 else 0.0
    return mat


if gene_sets_by_strat_imp:
    _ordered_keys = [
        (s, i)
        for s in ALL_STRATS
        for i in ["strict", "knn", "softimpute"]
        if (s, i) in gene_sets_by_strat_imp
    ]
    _set_list = [gene_sets_by_strat_imp[k] for k in _ordered_keys]
    _jmat = jaccard_matrix(_set_list)
    _jlabels = [f"{s}\n{i}" for s, i in _ordered_keys]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(_jmat, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(_jlabels)))
    ax.set_yticks(range(len(_jlabels)))
    ax.set_xticklabels(_jlabels, rotation=90, fontsize=7)
    ax.set_yticklabels(_jlabels, fontsize=7)
    for i in range(len(_jlabels)):
        for j in range(len(_jlabels)):
            ax.text(j, i, f"{_jmat[i, j]:.2f}", ha="center", va="center", fontsize=5)
    plt.colorbar(im, ax=ax, label="Jaccard similarity")
    ax.set_title("Pairwise Jaccard gene set overlap by (strat, imp)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "gene_jaccard_overlap.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "gene_jaccard_overlap.png", bbox_inches="tight", dpi=200)
    plt.show()

In [ ]:
# ── §7.3 Set intersection: supervenn plots ────────────────────────────────────

try:
    from supervenn import supervenn

    # Figure 1: fixed imp="strict", varying strat (how harshness shrinks gene universe)
    _sv_strats = [s for s in ALL_STRATS if (s, "strict") in gene_sets_by_strat_imp]
    if len(_sv_strats) >= 2:
        _sv_sets = [gene_sets_by_strat_imp[(s, "strict")] for s in _sv_strats]
        fig_sv1, ax_sv1 = plt.subplots(figsize=(14, 5))
        supervenn(_sv_sets, _sv_strats, ax=ax_sv1)
        ax_sv1.set_title("Gene overlap across strategies (imp=strict)")
        fig_sv1.tight_layout()
        fig_sv1.savefig(FIGURES_DIR / "supervenn_by_strat.svg", bbox_inches="tight")
        fig_sv1.savefig(
            FIGURES_DIR / "supervenn_by_strat.png", bbox_inches="tight", dpi=200
        )
        plt.show()
        core_genes = set.intersection(*_sv_sets)
        print(f"Core genes (all strategies): {len(core_genes)}")

    # Figure 2: fixed strat="A_confirmed_bad", varying imp
    _sv_imps = ["strict", "knn", "softimpute"]
    _sv_imp_sets = [
        gene_sets_by_strat_imp.get(("A_confirmed_bad", i), set()) for i in _sv_imps
    ]
    _sv_imp_sets_nonempty = [
        (i, s) for i, s in zip(_sv_imps, _sv_imp_sets) if len(s) > 0
    ]
    if len(_sv_imp_sets_nonempty) >= 2:
        _imp_labels, _imp_sets = zip(*_sv_imp_sets_nonempty)
        fig_sv2, ax_sv2 = plt.subplots(figsize=(10, 4))
        supervenn(list(_imp_sets), list(_imp_labels), ax=ax_sv2)
        ax_sv2.set_title(
            "Gene overlap across imputation methods (strat=A_confirmed_bad)"
        )
        fig_sv2.tight_layout()
        fig_sv2.savefig(FIGURES_DIR / "supervenn_by_imp.svg", bbox_inches="tight")
        fig_sv2.savefig(
            FIGURES_DIR / "supervenn_by_imp.png", bbox_inches="tight", dpi=200
        )
        plt.show()

except ImportError:
    print("supervenn not installed — run: pip install supervenn")

---
## §8 — Progressive Gene Accumulation / Depletion Curves

How does gene count change as sample removal becomes more aggressive? Track the progressive loss of genes across the harshness axis.

In [ ]:
# ── §8.1 Harshness ordering ───────────────────────────────────────────────────
# S0=permissive → G=aggressive. Within each strat: strict < knn < softimpute
# (strict loses genes; softimpute recovers the most).

HARSHNESS_ORDER = [
    ("S0_no_removal", "strict"),
    ("S0_no_removal", "knn"),
    ("S0_no_removal", "softimpute"),
    ("A_confirmed_bad", "strict"),
    ("A_confirmed_bad", "knn"),
    ("A_confirmed_bad", "softimpute"),
    ("B_extended_bad", "strict"),
    ("B_extended_bad", "knn"),
    ("B_extended_bad", "softimpute"),
    ("C_rnaseq_only", "strict"),
    ("C_rnaseq_only", "knn"),
    ("C_rnaseq_only", "softimpute"),
    ("D_malignant_only", "strict"),
    ("D_malignant_only", "knn"),
    ("D_malignant_only", "softimpute"),
    ("E1_iterative_r1", "strict"),
    ("E1_iterative_r1", "knn"),
    ("E1_iterative_r1", "softimpute"),
    ("E2_iterative_r2", "strict"),
    ("E2_iterative_r2", "knn"),
    ("E2_iterative_r2", "softimpute"),
    ("E3_iterative_r3", "strict"),
    ("E3_iterative_r3", "knn"),
    ("E3_iterative_r3", "softimpute"),
    ("F_microarray_only", "strict"),
    ("F_microarray_only", "knn"),
    ("F_microarray_only", "softimpute"),
    ("G_affymetrix_only", "strict"),
    ("G_affymetrix_only", "knn"),
    ("G_affymetrix_only", "softimpute"),
]
# Filter to only combinations that exist in gene_sets_by_strat_imp.
HARSHNESS_ORDER = [si for si in HARSHNESS_ORDER if si in gene_sets_by_strat_imp]

angel_representative_genes = gene_sets_angel.get(("S0_no_removal", "strict"), set())
print(f"HARSHNESS_ORDER steps: {len(HARSHNESS_ORDER)}")
print(f"Angel gene set size (reference): {len(angel_representative_genes)}")

In [ ]:
# ── §8.2 Gene retention curve ─────────────────────────────────────────────────

if HARSHNESS_ORDER:
    _step_counts = [len(gene_sets_by_strat_imp[k]) for k in HARSHNESS_ORDER]
    _cumulative_union: list[int] = []
    _running_union: set[str] = set()
    for k in HARSHNESS_ORDER:
        _running_union |= gene_sets_by_strat_imp[k]
        _cumulative_union.append(len(_running_union))

    _x = range(len(HARSHNESS_ORDER))
    _labels = [f"{s}\n{i}" for s, i in HARSHNESS_ORDER]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

    ax1.plot(_x, _step_counts, marker="o", markersize=4, label="n_genes (per step)")
    ax1.plot(
        _x,
        _cumulative_union,
        marker="s",
        markersize=4,
        linestyle="--",
        label="cumulative union",
    )
    if angel_representative_genes:
        ax1.axhline(
            len(angel_representative_genes),
            color="red",
            linestyle=":",
            label=f"25_angel ({len(angel_representative_genes)} genes)",
        )
    ax1.set_xticks(list(_x))
    ax1.set_xticklabels(_labels, rotation=45, ha="right", fontsize=7)
    ax1.set_ylabel("Number of genes")
    ax1.set_title("Gene retention along harshness axis")
    ax1.legend(fontsize=8)

    _prev_counts = [_step_counts[0]] + _step_counts[:-1]
    _lost = [max(0, p - c) for p, c in zip(_prev_counts, _step_counts)]
    _retained = _step_counts
    ax2.bar(_x, _retained, label="retained", color="#4292c6")
    ax2.bar(_x, _lost, bottom=_retained, label="lost vs prev", color="#d62728")
    ax2.set_xticks(list(_x))
    ax2.set_xticklabels(_labels, rotation=45, ha="right", fontsize=7)
    ax2.set_ylabel("Genes")
    ax2.set_title("Genes retained vs lost per step")
    ax2.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "gene_retention_curve.svg", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "gene_retention_curve.png", bbox_inches="tight", dpi=200)
    plt.show()
else:
    print("HARSHNESS_ORDER is empty — run §6 first.")

In [ ]:
# ── §8.3 Imputation recovery supervenn (strat=A_confirmed_bad) ────────────────

try:
    from supervenn import supervenn

    _imps = ["strict", "knn", "softimpute"]
    _imp_sets_abc = {
        i: gene_sets_by_strat_imp.get(("A_confirmed_bad", i), set()) for i in _imps
    }
    _nonempty = {i: s for i, s in _imp_sets_abc.items() if len(s) > 0}

    if len(_nonempty) >= 2:
        _labels_n = list(_nonempty.keys())
        _sets_n = list(_nonempty.values())
        fig_imp, ax_imp = plt.subplots(figsize=(10, 4))
        supervenn(_sets_n, _labels_n, ax=ax_imp)
        ax_imp.set_title("Gene recovery by imputation method (strat=A_confirmed_bad)")
        fig_imp.tight_layout()
        fig_imp.savefig(FIGURES_DIR / "supervenn_imp_recovery.svg", bbox_inches="tight")
        fig_imp.savefig(
            FIGURES_DIR / "supervenn_imp_recovery.png", bbox_inches="tight", dpi=200
        )
        plt.show()

        if "strict" in _nonempty and "knn" in _nonempty:
            recovered_knn = _nonempty["knn"] - _nonempty["strict"]
            print(f"Genes recovered by KNN vs strict: {len(recovered_knn)}")
        if "strict" in _nonempty and "softimpute" in _nonempty:
            recovered_soft = _nonempty["softimpute"] - _nonempty["strict"]
            print(f"Genes recovered by softimpute vs strict: {len(recovered_soft)}")
    else:
        print("Not enough non-empty imputation sets for supervenn.")

except ImportError:
    print("supervenn not installed — run: pip install supervenn")

---
## §9 — GO Enrichment Analysis Setup

Load the Gene Ontology database and define the enrichment function used in §10 and §11.

In [ ]:
import os

GO_OBO_PATH = "go-basic.obo"
GO_GAF_PATH = "goa_human.gaf"
GO_GAF_GZ = "goa_human.gaf.gz"

# Check for existing files in current dir or BG project cache
_BG_CACHE = "../../Retroelements/T2T_genes_article/T2T_transposons_genes/"


def _find_or_download(fname: str, url: str, gz_name: str | None = None) -> str:
    for candidate in [fname, os.path.join(_BG_CACHE, fname)]:
        if os.path.exists(candidate):
            print(f"Found: {candidate}")
            return candidate
    import urllib.request

    print(f"Downloading {url} ...")
    tmp = gz_name or fname
    urllib.request.urlretrieve(url, tmp)
    if gz_name:
        import gzip as _gz
        import shutil

        with _gz.open(gz_name, "rb") as f_in, open(fname, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz_name)
    print(f"Saved: {fname}")
    return fname


GO_OBO_PATH = _find_or_download(
    GO_OBO_PATH,
    "http://purl.obolibrary.org/obo/go/go-basic.obo",
)
GO_GAF_PATH = _find_or_download(
    GO_GAF_PATH,
    "http://geneontology.org/gene-associations/goa_human.gaf.gz",
    gz_name=GO_GAF_GZ,
)
print("GO database files ready.")

In [ ]:
def load_go_database(
    obo_path: str, gaf_path: str
) -> tuple[object, dict[str, set[str]], list[str]]:
    """
    Load the Gene Ontology DAG and human gene → GO term associations.

    The background gene list is fixed to ALL genes with GO annotations in the GAF
    file, not a dataset-specific subset. A fixed background ensures that enrichment
    strength correctly increases when more biologically relevant genes are retained.

    Parameters
    ----------
    obo_path : str
        Path to go-basic.obo file.
    gaf_path : str
        Path to goa_human.gaf file (not compressed).

    Returns
    -------
    godag : GODag
    full_assoc : dict[str, set[str]]
        Mapping gene symbol → set of GO IDs.
    GO_BACKGROUND : list[str]
        All gene symbols present in the GAF file.
    """
    from goatools.anno.gaf_reader import GafReader
    from goatools.obo_parser import GODag

    godag = GODag(obo_path, optional_attrs={"relationship"})
    gaf = GafReader(gaf_path)
    ns2assoc = gaf.get_ns2assc()
    full_assoc: dict[str, set[str]] = {}
    for ns_assoc in ns2assoc.values():
        for gene, go_ids in ns_assoc.items():
            full_assoc.setdefault(gene, set()).update(go_ids)
    GO_BACKGROUND = sorted(full_assoc.keys())
    print(f"GO DAG loaded: {len(godag)} terms")
    print(f"Annotated genes: {len(GO_BACKGROUND)}")
    return godag, full_assoc, GO_BACKGROUND


try:
    godag, full_assoc, GO_BACKGROUND = load_go_database(GO_OBO_PATH, GO_GAF_PATH)
except Exception as exc:
    godag, full_assoc, GO_BACKGROUND = None, {}, []
    print(f"GO database load failed: {exc}")

In [ ]:
def run_goatools_enrichment(
    study_genes: list[str],
    background_genes: list[str],
    go_assoc: dict[str, set[str]],
    go_dag: object,
    alpha: float = 0.05,
    methods: list[str] | None = None,
    min_study_count: int = 3,
) -> pd.DataFrame:
    """
    Run GO term enrichment using goatools Fisher's exact test.

    Parameters
    ----------
    study_genes : list of str
        Gene symbols to test for enrichment.
    background_genes : list of str
        Full background gene universe.
    go_assoc : dict[str, set[str]]
        Gene → GO term mapping.
    go_dag : GODag
        Loaded GO DAG.
    alpha : float
        FDR threshold for significance.
    methods : list of str or None
        Multiple testing correction methods (default: ["fdr_bh"]).
    min_study_count : int
        Minimum number of study genes per GO term.

    Returns
    -------
    pd.DataFrame
        Enrichment results with columns: GO_ID, name, namespace, p_uncorrected,
        p_fdr_bh, ratio_in_study, ratio_in_pop, study_genes_str.
    """
    from goatools.go_enrichment import GOEnrichmentStudy

    if methods is None:
        methods = ["fdr_bh"]
    if go_dag is None or not background_genes:
        return pd.DataFrame()

    goeaobj = GOEnrichmentStudy(
        background_genes,
        go_assoc,
        go_dag,
        propagate_counts=True,
        alpha=alpha,
        methods=methods,
    )
    results = goeaobj.run_study(study_genes, prt=None)
    rows = []
    for r in results:
        if r.study_count < min_study_count:
            continue
        p_fdr = getattr(r, "p_fdr_bh", 1.0)
        rows.append(
            {
                "GO_ID": r.GO,
                "name": r.name,
                "namespace": r.NS,
                "p_uncorrected": r.p_uncorrected,
                "p_fdr_bh": p_fdr,
                "ratio_in_study": f"{r.study_count}/{r.study_n}",
                "ratio_in_pop": f"{r.pop_count}/{r.pop_n}",
                "study_genes_str": ",".join(sorted(r.study_items)),
            }
        )
    df_go = pd.DataFrame(rows)
    if len(df_go):
        df_go = df_go.sort_values("p_fdr_bh")
    return df_go


print("run_goatools_enrichment defined.")

---
## §10 — Progressive GO Term Accumulation Curves

As sample removal strategy becomes more aggressive, which GO terms are gained or lost? Track GO enrichment at each step of the harshness axis.

In [ ]:
# ── §10.1 Harshness-ordered gene sets for GO analysis ─────────────────────────
# For each step in HARSHNESS_ORDER, get the union of gene sets across all methods
# (except 25_angel), intersected with GO-annotated genes.

if GO_BACKGROUND and HARSHNESS_ORDER:
    _go_bg_set = set(GO_BACKGROUND)
    _go_step_genes = {
        si: gene_sets_by_strat_imp[si] & _go_bg_set for si in HARSHNESS_ORDER
    }
    _angel_go_genes = angel_representative_genes & _go_bg_set
    print(f"GO-annotated gene counts per step:")
    for si, gs in list(_go_step_genes.items())[:5]:
        print(f"  {si}: {len(gs)} genes")
    print(f"  ... (total {len(_go_step_genes)} steps)")
    print(f"Angel GO-annotated genes: {len(_angel_go_genes)}")
else:
    _go_step_genes = {}
    _angel_go_genes = set()
    print("Skipped: GO database or HARSHNESS_ORDER not available.")

In [ ]:
GO_CACHE_DIR = Path("go_cache")
GO_CACHE_DIR.mkdir(exist_ok=True)

go_results: dict[tuple[str, str], pd.DataFrame] = {}

if godag is not None and _go_step_genes:
    for strat, imp in HARSHNESS_ORDER:
        cache_path = GO_CACHE_DIR / f"{strat}_{imp}.parquet"
        if cache_path.exists():
            go_results[(strat, imp)] = pd.read_parquet(cache_path)
            continue
        study = list(_go_step_genes[(strat, imp)])
        df_enr = run_goatools_enrichment(study, GO_BACKGROUND, full_assoc, godag)
        df_enr.to_parquet(cache_path, index=False)
        go_results[(strat, imp)] = df_enr
        print(f"  {strat} / {imp}: {len(df_enr)} enriched terms (FDR<0.05)")

    _angel_cache = GO_CACHE_DIR / "angel_reference.parquet"
    if _angel_cache.exists():
        go_results_angel = pd.read_parquet(_angel_cache)
    elif _angel_go_genes:
        go_results_angel = run_goatools_enrichment(
            list(_angel_go_genes), GO_BACKGROUND, full_assoc, godag
        )
        go_results_angel.to_parquet(_angel_cache, index=False)
    else:
        go_results_angel = pd.DataFrame()
    print(f"Angel reference: {len(go_results_angel)} enriched terms")
else:
    go_results_angel = pd.DataFrame()
    print("Skipped: GO database not available.")

In [ ]:
# ── §10.3 GO accumulation curve ───────────────────────────────────────────────

if go_results:
    _sig_term_sets = {
        si: set(go_results[si].loc[go_results[si].p_fdr_bh < 0.05, "GO_ID"])
        for si in HARSHNESS_ORDER
        if si in go_results
    }
    _n_sig = [len(_sig_term_sets.get(si, set())) for si in HARSHNESS_ORDER]
    _cumul_terms: list[int] = []
    _seen_terms: set[str] = set()
    _new_per_step: list[int] = []
    _n_genes_list = [len(_go_step_genes.get(si, set())) for si in HARSHNESS_ORDER]
    for si in HARSHNESS_ORDER:
        new_terms = _sig_term_sets.get(si, set()) - _seen_terms
        _seen_terms |= _sig_term_sets.get(si, set())
        _cumul_terms.append(len(_seen_terms))
        _new_per_step.append(len(new_terms))

    _labels_10 = [f"{s[:6]}\n{i[:3]}" for s, i in HARSHNESS_ORDER]
    _x10 = list(range(len(HARSHNESS_ORDER)))

    fig10, (ax10a, ax10b, ax10c) = plt.subplots(1, 3, figsize=(21, 5))

    ax10a.plot(
        _x10, _n_genes_list, marker="o", markersize=4, label="GO-annotated genes"
    )
    ax10a.plot(
        _x10,
        _cumul_terms,
        marker="s",
        markersize=4,
        linestyle="--",
        label="cumulative unique GO terms",
    )
    if _angel_go_genes:
        ax10a.axhline(
            len(_angel_go_genes),
            color="red",
            linestyle=":",
            label=f"Angel genes ({len(_angel_go_genes)})",
        )
    if len(go_results_angel):
        _n_angel_terms = len(go_results_angel.loc[go_results_angel.p_fdr_bh < 0.05])
        ax10a.axhline(
            _n_angel_terms,
            color="darkred",
            linestyle="-.",
            label=f"Angel GO terms ({_n_angel_terms})",
        )
    ax10a.set_xticks(_x10)
    ax10a.set_xticklabels(_labels_10, rotation=45, ha="right", fontsize=7)
    ax10a.set_ylabel("Count")
    ax10a.set_title("Gene count and GO term accumulation")
    ax10a.legend(fontsize=7)

    ax10b.bar(_x10, _new_per_step, color="#4292c6")
    ax10b.set_xticks(_x10)
    ax10b.set_xticklabels(_labels_10, rotation=45, ha="right", fontsize=7)
    ax10b.set_ylabel("New unique GO terms")
    ax10b.set_title("New GO terms per step")

    _density = [
        n / max(1, g - (_n_genes_list[i - 1] if i > 0 else 0))
        for i, (n, g) in enumerate(zip(_new_per_step, _n_genes_list))
    ]
    ax10c.bar(_x10, _density, color="#41ab5d")
    ax10c.set_xticks(_x10)
    ax10c.set_xticklabels(_labels_10, rotation=45, ha="right", fontsize=7)
    ax10c.set_ylabel("New GO terms / genes added")
    ax10c.set_title("Biological information density per gene added")

    fig10.tight_layout()
    fig10.savefig(FIGURES_DIR / "go_accumulation_curve.svg", bbox_inches="tight")
    fig10.savefig(
        FIGURES_DIR / "go_accumulation_curve.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("GO results not available — run §10.2 first.")

In [ ]:
# ── §10.4 Terms lost between adjacent strategy pairs ─────────────────────────

if len(HARSHNESS_ORDER) >= 2 and go_results:
    _loss_data = []
    for i in range(1, len(HARSHNESS_ORDER)):
        prev = HARSHNESS_ORDER[i - 1]
        curr = HARSHNESS_ORDER[i]
        prev_terms = _sig_term_sets.get(prev, set())
        curr_terms = _sig_term_sets.get(curr, set())
        lost = prev_terms - curr_terms
        _loss_data.append(
            {
                "step": i,
                "prev": prev,
                "curr": curr,
                "n_lost": len(lost),
                "lost_terms": lost,
            }
        )

    if _loss_data:
        max_loss_step = max(_loss_data, key=lambda d: d["n_lost"])
        print(
            f"Largest GO term loss: {max_loss_step['prev']} → {max_loss_step['curr']}: {max_loss_step['n_lost']} terms"
        )

        _lost_df = pd.DataFrame(_loss_data)[["step", "n_lost"]]
        fig_lost, ax_lost = plt.subplots(figsize=(12, 4))
        ax_lost.bar(_lost_df.step, _lost_df.n_lost, color="#d62728")
        ax_lost.set_xlabel("Step index")
        ax_lost.set_ylabel("GO terms lost vs previous step")
        ax_lost.set_title("GO term loss between adjacent harshness steps")
        fig_lost.tight_layout()
        fig_lost.savefig(
            FIGURES_DIR / "go_lost_terms_per_step.svg", bbox_inches="tight"
        )
        fig_lost.savefig(
            FIGURES_DIR / "go_lost_terms_per_step.png", bbox_inches="tight", dpi=200
        )
        plt.show()
else:
    print("Insufficient data for GO term loss analysis.")

In [ ]:
# ── §10.5 Imputation-specific term recovery: supervenn ────────────────────────

try:
    from supervenn import supervenn

    _imp_term_sets = {
        i: set(
            go_results[("A_confirmed_bad", i)].loc[
                go_results[("A_confirmed_bad", i)].p_fdr_bh < 0.05, "GO_ID"
            ]
        )
        for i in ["strict", "knn", "softimpute"]
        if ("A_confirmed_bad", i) in go_results
    }
    _nonempty_imp = {k: v for k, v in _imp_term_sets.items() if v}
    if len(_nonempty_imp) >= 2:
        fig_sv_go, ax_sv_go = plt.subplots(figsize=(10, 5))
        supervenn(list(_nonempty_imp.values()), list(_nonempty_imp.keys()), ax=ax_sv_go)
        ax_sv_go.set_title(
            "GO term recovery by imputation (strat=A_confirmed_bad, FDR<0.05)"
        )
        fig_sv_go.tight_layout()
        fig_sv_go.savefig(FIGURES_DIR / "supervenn_go_by_imp.svg", bbox_inches="tight")
        fig_sv_go.savefig(
            FIGURES_DIR / "supervenn_go_by_imp.png", bbox_inches="tight", dpi=200
        )
        plt.show()
    else:
        print("Not enough GO result sets for supervenn.")

except ImportError:
    print("supervenn not installed.")

---
## §11 — FL-Specific Gene Retention Analysis

Which Follicular Lymphoma marker genes are retained or lost across harmonization strategies? What is the biological cost of aggressive sample removal?

## Gene Set Sources and Citations

| Gene set | Source | DOI / URL |
|---|---|---|
| FL transcriptomic subtypes C1/C2/C3 (FL_2025_MARKERS) | Xochelli et al. 2025, *Leukemia* | https://doi.org/10.1038/s41375-025-02603-9 |
| legacy internal GC B-cell markers (Centroblast/Centrocyte) | unpublished internal panel | superseded by the Dybkaer et al. 2015 panels |
| FL PFS prognostic signature (FL_PROG_SIGNATURES) | Pastore et al. 2019, *J Clin Oncol* | https://doi.org/10.1200/JCO.18.01545 (PMID 29475724) |
| POD24 mutation markers | Huet et al. 2018, *Blood* | https://doi.org/10.1182/blood-2018-03-837443 |
| PROGENy pathway activating genes | Schubert et al. 2018, *Nature Communications* | https://doi.org/10.1038/s41467-017-02391-6 |
| Housekeeping genes (stability reference) | Eisenberg & Levanon 2013, *Trends Genet* | https://doi.org/10.1016/j.tig.2013.05.010 |

In [ ]:
# ── §11.1 Compile FL gene list ────────────────────────────────────────────────

# Source 1: Xochelli et al. 2025 FL C1/C2/C3 subtype markers
FL_2025_MARKERS = [
    "BCL2",
    "BCL6",
    "MKI67",
    "PCNA",
    "AICDA",
    "IRF4",
    "FOXP1",
    "MYC",
    "CD10",
    "CD19",
    "CD20",
    "CD21",
    "CD23",
    "CD38",
    "CD44",
    "CD79A",
    "CXCR4",
    "CXCR5",
    "EZH2",
    "FAS",
    "IGHM",
    "IGHD",
    "IGHG",
    "LMO2",
    "MME",
    "MYBL1",
    "PAX5",
    "PRDM1",
    "SPIB",
    "XBP1",
]

# Source 2: legacy internal GC B-cell markers (Centroblast / Centrocyte)
LEGACY_GC_MARKERS = [
    "AICDA",
    "BCL6",
    "CXCR4",
    "MYBL1",
    "TOP2A",
    "MKI67",
    "PCNA",
    "LMO2",
    "CXCR5",
    "CD83",
    "EBI3",
    "HLA-DRA",
    "CCND2",
    "RGS13",
]

# Source 3: FL PFS prognostic signature (Pastore et al. 2019)
FL_PROG_SIGNATURES = [
    "BCL2L1",
    "CALCRL",
    "CCDC50",
    "DPM2",
    "FAS",
    "GADD45B",
    "GFI1",
    "HVCN1",
    "LBH",
    "MARCKS",
    "PCNA",
    "PER2",
    "PTGDS",
    "RAB27A",
    "S100A8",
    "S1PR1",
    "SGK1",
    "SLC2A6",
    "SPIB",
    "ZBTB7A",
]

# Source 4: POD24 mutation markers (Huet et al. 2018)
POD24_MARKERS = [
    "CREBBP",
    "EP300",
    "EZH2",
    "KMT2D",
    "MEF2B",
    "MYD88",
    "TP53",
    "VHL",
    "ARID1A",
    "CARD11",
    "CDKN2A",
    "FOXO1",
    "GNA13",
    "HIST1H1E",
]

FL_GENES_ALL = sorted(
    set(FL_2025_MARKERS)
    | set(LEGACY_GC_MARKERS)
    | set(FL_PROG_SIGNATURES)
    | set(POD24_MARKERS)
)
FL_GENE_CATEGORIES = {
    g: (
        "FL_2025"
        if g in FL_2025_MARKERS
        else (
            "Normal_B_cells"
            if g in LEGACY_GC_MARKERS
            else "FL_PROG" if g in FL_PROG_SIGNATURES else "POD24"
        )
    )
    for g in FL_GENES_ALL
}

print(f"FL_GENES_ALL: {len(FL_GENES_ALL)} unique genes")
for cat in ["FL_2025", "Normal_B_cells", "FL_PROG", "POD24"]:
    n = sum(1 for g, c in FL_GENE_CATEGORIES.items() if c == cat)
    print(f"  {cat}: {n}")


def validate_gene_aliases(
    gene_list: list[str], reference_genes: set[str]
) -> dict[str, str]:
    """
    Check which genes from a curated list are absent from the expression matrices
    and propose HGNC alias resolution.

    Parameters
    ----------
    gene_list : list of str
        Curated gene symbols to validate.
    reference_genes : set of str
        Union of all gene_sets values (not 25_angel).

    Returns
    -------
    dict[str, str]
        Mapping absent_symbol → suggested_replacement.
    """
    KNOWN_ALIASES = {"CD10": "MME", "CD95": "FAS", "TNFRSF6": "FAS", "CD23": "FCER2"}
    missing = {}
    for g in gene_list:
        if g not in reference_genes:
            replacement = KNOWN_ALIASES.get(g)
            missing[g] = replacement or "not found"
    return missing


if gene_sets_by_strat_imp:
    _ref_all_genes = set.union(*gene_sets_by_strat_imp.values())
    _aliases = validate_gene_aliases(FL_GENES_ALL, _ref_all_genes)
    if _aliases:
        print(f"\nGenes absent from expression matrices ({len(_aliases)}):")
        for g, repl in sorted(_aliases.items()):
            print(f"  {g} → {repl}")
    else:
        print("\nAll FL genes found in expression matrices.")
else:
    _ref_all_genes = set(FL_GENES_ALL)
    print("gene_sets_by_strat_imp empty — skipping alias validation.")

In [ ]:
# ── §11.2 FL gene presence matrix ─────────────────────────────────────────────

if gene_sets_by_strat_imp:
    _fl_presence_rows = []
    for g in FL_GENES_ALL:
        row = {
            f"{s}__{i}": int(g in gene_sets_by_strat_imp.get((s, i), set()))
            for s in ALL_STRATS
            for i in ["strict", "knn", "softimpute"]
            if (s, i) in gene_sets_by_strat_imp
        }
        row["gene"] = g
        row["category"] = FL_GENE_CATEGORIES.get(g, "other")
        _fl_presence_rows.append(row)

    fl_presence = pd.DataFrame(_fl_presence_rows).set_index("gene")
    category_col = fl_presence.pop("category")
    fl_presence.insert(0, "category", category_col)

    attempt_cols = [c for c in fl_presence.columns if c != "category"]
    print(f"fl_presence shape: {fl_presence.shape}")
    print(fl_presence[attempt_cols].sum(axis=0).describe())
else:
    fl_presence = pd.DataFrame()
    print("gene_sets_by_strat_imp empty — skipping §11.2.")

In [ ]:
# ── §11.3 FL gene retention heatmap ──────────────────────────────────────────

if not fl_presence.empty:
    _attempt_cols_sorted = sorted(
        [c for c in fl_presence.columns if c != "category"],
        key=lambda c: (
            (
                ALL_STRATS.index(c.split("__")[0])
                if c.split("__")[0] in ALL_STRATS
                else 99
            ),
            (
                ["strict", "knn", "softimpute"].index(c.split("__")[-1])
                if c.split("__")[-1] in ["strict", "knn", "softimpute"]
                else 9
            ),
        ),
    )
    _gene_order = fl_presence.sort_values("category").index.tolist()

    fig_fl, ax_fl = plt.subplots(
        figsize=(
            max(16, len(_attempt_cols_sorted) * 0.5),
            max(8, len(_gene_order) * 0.3),
        )
    )
    _data_plot = fl_presence.loc[_gene_order, _attempt_cols_sorted].astype(float)
    sns.heatmap(
        _data_plot,
        ax=ax_fl,
        cmap=["#ffcccc", "#006400"],
        linewidths=0.3,
        linecolor="#eeeeee",
        cbar_kws={"label": "Gene present"},
        xticklabels=True,
        yticklabels=True,
    )
    ax_fl.set_xticklabels(ax_fl.get_xticklabels(), rotation=90, fontsize=6)
    ax_fl.set_yticklabels(ax_fl.get_yticklabels(), fontsize=7)

    # Category color bar on the left
    _cat_colors = {
        "FL_2025": "#6a0dad",
        "Normal_B_cells": "#41ab5d",
        "FL_PROG": "#ff7f0e",
        "POD24": "#d62728",
    }
    for ytick in ax_fl.get_yticklabels():
        gene = ytick.get_text()
        cat = FL_GENE_CATEGORIES.get(gene, "other")
        ytick.set_color(_cat_colors.get(cat, "#000000"))

    ax_fl.set_title("FL gene presence across harmonization attempts (dark=present)")
    fig_fl.tight_layout()
    fig_fl.savefig(FIGURES_DIR / "fl_retention_heatmap.svg", bbox_inches="tight")
    fig_fl.savefig(
        FIGURES_DIR / "fl_retention_heatmap.png", bbox_inches="tight", dpi=200
    )
    plt.show()

In [ ]:
# ── §11.4 FL gene progressive retention curves ────────────────────────────────

if HARSHNESS_ORDER and gene_sets_by_strat_imp:
    _cat_list = sorted(set(FL_GENE_CATEGORIES.values()))
    _x11 = list(range(len(HARSHNESS_ORDER)))
    _labels11 = [f"{s[:6]}\n{i[:3]}" for s, i in HARSHNESS_ORDER]

    # Figure A: one subplot per FL gene category
    fig_a, axes_a = plt.subplots(
        1, len(_cat_list), figsize=(5 * len(_cat_list), 5), sharey=True
    )
    if len(_cat_list) == 1:
        axes_a = [axes_a]
    for ax_cat, cat in zip(axes_a, _cat_list):
        _cat_genes = [g for g, c in FL_GENE_CATEGORIES.items() if c == cat]
        _fracs = [
            sum(1 for g in _cat_genes if g in gene_sets_by_strat_imp.get(si, set()))
            / max(1, len(_cat_genes))
            for si in HARSHNESS_ORDER
        ]
        ax_cat.plot(
            _x11,
            _fracs,
            marker="o",
            markersize=4,
            color=_cat_colors.get(cat, "#AAAAAA"),
        )
        if angel_representative_genes:
            _angel_frac = sum(
                1 for g in _cat_genes if g in angel_representative_genes
            ) / max(1, len(_cat_genes))
            ax_cat.axhline(
                _angel_frac,
                color="red",
                linestyle=":",
                label=f"Angel ({_angel_frac:.2f})",
            )
            ax_cat.legend(fontsize=7)
        ax_cat.set_title(f"{cat} ({len(_cat_genes)} genes)")
        ax_cat.set_xticks(_x11)
        ax_cat.set_xticklabels(_labels11, rotation=45, ha="right", fontsize=7)
        ax_cat.set_ylim(0, 1.05)
        ax_cat.set_ylabel("Fraction of genes retained")
    fig_a.suptitle("FL gene retention by category along harshness axis", fontsize=12)
    fig_a.tight_layout()
    fig_a.savefig(
        FIGURES_DIR / "fl_retention_curves_by_category.svg", bbox_inches="tight"
    )
    fig_a.savefig(
        FIGURES_DIR / "fl_retention_curves_by_category.png",
        bbox_inches="tight",
        dpi=200,
    )
    plt.show()

    # Figure B: overall retention + per-category thin lines
    fig_b, ax_b11 = plt.subplots(figsize=(14, 5))
    _overall_fracs = [
        sum(1 for g in FL_GENES_ALL if g in gene_sets_by_strat_imp.get(si, set()))
        / max(1, len(FL_GENES_ALL))
        for si in HARSHNESS_ORDER
    ]
    ax_b11.plot(
        _x11, _overall_fracs, linewidth=2.5, color="#000000", label="All FL genes"
    )
    for cat in _cat_list:
        _cat_genes = [g for g, c in FL_GENE_CATEGORIES.items() if c == cat]
        _fracs_cat = [
            sum(1 for g in _cat_genes if g in gene_sets_by_strat_imp.get(si, set()))
            / max(1, len(_cat_genes))
            for si in HARSHNESS_ORDER
        ]
        ax_b11.plot(
            _x11,
            _fracs_cat,
            linewidth=0.8,
            alpha=0.6,
            color=_cat_colors.get(cat, "#AAAAAA"),
            label=cat,
        )
    if angel_representative_genes:
        _af = sum(1 for g in FL_GENES_ALL if g in angel_representative_genes) / max(
            1, len(FL_GENES_ALL)
        )
        ax_b11.axhline(_af, color="red", linestyle=":", label=f"Angel ({_af:.2f})")
    ax_b11.set_xticks(_x11)
    ax_b11.set_xticklabels(_labels11, rotation=45, ha="right", fontsize=7)
    ax_b11.set_ylim(0, 1.05)
    ax_b11.set_ylabel("Fraction of FL genes retained")
    ax_b11.set_title("FL gene retention across harshness axis")
    ax_b11.legend(fontsize=8, loc="lower left")
    fig_b.tight_layout()
    fig_b.savefig(FIGURES_DIR / "fl_retention_curves_overall.svg", bbox_inches="tight")
    fig_b.savefig(
        FIGURES_DIR / "fl_retention_curves_overall.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("HARSHNESS_ORDER empty — run §8.1 and §6 first.")

In [ ]:
# ── §11.5 GO enrichment in FL-specific gene subset ────────────────────────────

if godag is not None and gene_sets_by_strat_imp:
    _fl_go_results: dict[tuple[str, str], pd.DataFrame] = {}
    for si in HARSHNESS_ORDER[:6]:  # top-6 steps for tractability
        strat, imp = si
        study = [g for g in FL_GENES_ALL if g in gene_sets_by_strat_imp.get(si, set())]
        if len(study) >= 3:
            _cache_fl = GO_CACHE_DIR / f"fl_{strat}_{imp}.parquet"
            if _cache_fl.exists():
                _fl_go_results[si] = pd.read_parquet(_cache_fl)
            else:
                df_fl_enr = run_goatools_enrichment(
                    study, GO_BACKGROUND, full_assoc, godag
                )
                df_fl_enr.to_parquet(_cache_fl, index=False)
                _fl_go_results[si] = df_fl_enr

    if _fl_go_results:
        _fl_go_sig = {
            si: df[df.p_fdr_bh < 0.05].head(20) for si, df in _fl_go_results.items()
        }
        _all_fl_terms = (
            pd.concat(list(_fl_go_sig.values())).drop_duplicates("GO_ID")
            if any(len(v) for v in _fl_go_sig.values())
            else pd.DataFrame()
        )
        if len(_all_fl_terms):
            for ns in ["biological_process", "molecular_function"]:
                _ns_terms = _all_fl_terms[
                    _all_fl_terms.namespace.str.contains(ns[:2], case=False)
                ]
                if not len(_ns_terms):
                    continue
                _term_names = _ns_terms.set_index("GO_ID")["name"].to_dict()
                _dot_data = (
                    pd.DataFrame(
                        {
                            si: _fl_go_results[si]
                            .set_index("GO_ID")
                            .get("p_fdr_bh", pd.Series(dtype=float))
                            for si in list(_fl_go_results.keys())
                        }
                    )
                    .loc[
                        [
                            g
                            for g in _ns_terms.GO_ID
                            if g
                            in pd.concat(list(_fl_go_results.values())).GO_ID.values
                        ]
                    ]
                    .fillna(1.0)
                )

                if not _dot_data.empty:
                    fig_dot, ax_dot = plt.subplots(
                        figsize=(10, max(4, len(_dot_data) * 0.4))
                    )
                    sns.heatmap(
                        -np.log10(_dot_data.astype(float) + 1e-10),
                        ax=ax_dot,
                        cmap="Blues",
                        cbar_kws={"label": "-log10(p_fdr_bh)"},
                        linewidths=0.3,
                    )
                    ax_dot.set_yticks(range(len(_dot_data.index)))
                    ax_dot.set_yticklabels(
                        [_term_names.get(g, g) for g in _dot_data.index], fontsize=7
                    )
                    ax_dot.set_xticklabels(
                        [f"{s[:6]}\n{i[:3]}" for s, i in _dot_data.columns], fontsize=7
                    )
                    ax_dot.set_title(f"FL gene GO enrichment — {ns.replace('_', ' ')}")
                    fig_dot.tight_layout()
                    fig_dot.savefig(
                        FIGURES_DIR / f"fl_go_dot_{ns[:2]}.svg", bbox_inches="tight"
                    )
                    fig_dot.savefig(
                        FIGURES_DIR / f"fl_go_dot_{ns[:2]}.png",
                        bbox_inches="tight",
                        dpi=200,
                    )
                    plt.show()
else:
    print("GO database or gene sets not available — skipping §11.5.")

In [ ]:
# ── §11.6 Progressive FL-genes GO accumulation curve ─────────────────────────

if _fl_go_results:
    _fl_sig_sets = {
        si: set(_fl_go_results[si].loc[_fl_go_results[si].p_fdr_bh < 0.05, "GO_ID"])
        for si in HARSHNESS_ORDER[:6]
        if si in _fl_go_results
    }
    _fl_term_counts = [len(_fl_sig_sets.get(si, set())) for si in HARSHNESS_ORDER[:6]]
    _fl_steps = list(range(len(HARSHNESS_ORDER[:6])))
    _fl_labels = [f"{s[:6]}\n{i[:3]}" for s, i in HARSHNESS_ORDER[:6]]

    fig_fl_go, ax_fl_go = plt.subplots(figsize=(12, 4))
    ax_fl_go.plot(_fl_steps, _fl_term_counts, marker="o", markersize=5)
    if len(go_results_angel) and "GO_ID" in go_results_angel.columns:
        _n_angel_fl = len(go_results_angel.loc[go_results_angel.p_fdr_bh < 0.05])
        ax_fl_go.axhline(
            _n_angel_fl,
            color="red",
            linestyle=":",
            label=f"Angel ({_n_angel_fl} terms)",
        )
        ax_fl_go.legend(fontsize=8)
    ax_fl_go.set_xticks(_fl_steps)
    ax_fl_go.set_xticklabels(_fl_labels, rotation=45, ha="right", fontsize=7)
    ax_fl_go.set_ylabel("Significant GO terms (FDR<0.05)")
    ax_fl_go.set_title("FL-gene GO enrichment along harshness axis")
    fig_fl_go.tight_layout()
    fig_fl_go.savefig(FIGURES_DIR / "fl_go_accumulation.svg", bbox_inches="tight")
    fig_fl_go.savefig(
        FIGURES_DIR / "fl_go_accumulation.png", bbox_inches="tight", dpi=200
    )
    plt.show()
else:
    print("FL GO results not available — run §11.5 first.")

In [ ]:
# ── §11.7 Final recommendation table ─────────────────────────────────────────

_rec_rows = []
for _, row in df_ok.sort_values("composite_score", ascending=False).iterrows():
    si = (row["strat"], row["imp"])
    _fl_ret = (
        sum(1 for g in FL_GENES_ALL if g in gene_sets_by_strat_imp.get(si, set()))
        / max(1, len(FL_GENES_ALL))
        if gene_sets_by_strat_imp
        else None
    )
    _rec_rows.append(
        {
            "strat": row["strat"],
            "imp": row["imp"],
            "method": row["method"],
            "post_rm": row["post_rm"],
            "composite_score": row.get("composite_score"),
            "score_batch_mixing": row.get("score_batch_mixing"),
            "score_bio_preservation": row.get("score_bio_preservation"),
            "r2_RNA_BATCH": row.get("r2_RNA_BATCH"),
            "kbet_acceptance_rate_RNA_BATCH": row.get("kbet_acceptance_rate_RNA_BATCH"),
            "n_genes": row.get("n_genes"),
            "fl_gene_retention": _fl_ret,
            "is_top10": row.name in top_n.index,
        }
    )

rec_df = pd.DataFrame(_rec_rows)
rec_df.insert(0, "rank", range(1, len(rec_df) + 1))

# Display styled
_fmt = {
    "composite_score": "{:.3f}",
    "score_batch_mixing": "{:.3f}",
    "score_bio_preservation": "{:.3f}",
    "r2_RNA_BATCH": "{:.3f}",
    "fl_gene_retention": "{:.2f}",
}
styled = (
    rec_df.head(30)
    .style.background_gradient(subset=["composite_score"], cmap="RdYlGn")
    .background_gradient(subset=["fl_gene_retention"], cmap="Blues")
    .format(_fmt, na_rep="N/A")
)
display(styled)

# Export CSV
rec_df.to_csv("recommendation_table.csv", index=False)
print("Saved: recommendation_table.csv")

# Upload to S3
try:
    _s3_client.put_object(
        Bucket=S3_BUCKET,
        Key=f"{S3_PREFIX}/recommendation_table.csv",
        Body=rec_df.to_csv(index=False).encode(),
    )
    print(f"Uploaded to s3://{S3_BUCKET}/{S3_PREFIX}/recommendation_table.csv")
except Exception as exc:
    print(f"S3 upload failed: {exc}")